# Adaptive A\* for Mobile-Robot Global Path Planning — Independent Re-Implementation and Reproduction Study

| | |
|---|---|
| **Student** | *Your name* — *Roll number* |
| **Course** | *Course name* — Assignment 1: Research Paper Re-Implementation |
| **Paper** | H. Cao, Z. Guo, Y. Zhang, Z. Lu, and L. Jiang, "An Adaptive A\* Algorithm for Mobile Robots Global Path Planning," *Electronics*, vol. 15, no. 13, Art. 2807, 2026, doi: 10.3390/electronics15132807 |
| **Source code of the paper** | Not publicly available; everything below was written independently from the paper |

The notebook runs top to bottom with **Run All**: CPU only, no Internet, no uploaded files.

## What is our own work in this notebook?

The paper gives only equations, a short description and result tables. **Everything below was produced by us:**

| # | Our contribution | Where |
|---|---|---|
| 1 | Read and analysed the paper: problem, method, data, feasibility and what is missing (assignment Q1–Q7) | Part 0 |
| 2 | **Wrote the whole implementation from scratch:** grid model, A\* (Algorithm 1), adaptive weight (Eq. 6), hybrid heuristic (Eq. 7), redundant point deletion (Sec. 3.3), Dijkstra, bidirectional A\*, an exact line-of-sight test, the metrics | Part 2 |
| 3 | Made and documented every decision the paper leaves open (17 missing details → explicit assumptions) | Parts 0 and 2 |
| 4 | Wrote correctness tests | Part 3 |
| 5 | **Digitised the paper's maps from its Fig. 5** (published only as a picture). Our baseline A\* and Dijkstra on these maps reproduce the paper's node counts almost exactly, which also revealed two settings the paper never states | Part 4 |
| 6 | **Audited the paper's published numbers** and found internal contradictions: e.g. several percentages in the text do not match its tables, two rows of Table 1 appear to be swapped, and a Table 5 start cell is an obstacle in its own figure | Parts 5 and 6 |
| 7 | Reproduced every experiment of Tables 1–5 and compared each row with the paper | Parts 6 and 8 |
| 8 | **Added experiments the paper does not have:** A\* + pruning and octile baselines, a paired timing study, 150 random maps with Wilcoxon tests, and an optimality check. We also tested a hypothesis about the authors' code: their published Adaptive A\* numbers are *consistent with* the hybrid heuristic **without** the ½ of Eq. 7 ("V4"), which is evidence, not proof | Parts 7, 9 and 10 |
| 9 | Critical analysis: comparable / improved / lower, with technical justification | Part 11 |

### Why does it run in a few minutes?
This is a **classical graph-search algorithm**, not deep learning: there is no training. One A\* search on a 30×30 to 100×100 grid takes about 0.5–15 ms in Python. The paper itself reports 0.01–1.93 s per search in MATLAB.

A full run of this notebook executes roughly **6,000 searches**:
- about 1,400 for Tables 1–5 (30 timing repeats each);
- 1,800 for the paired timing study;
- 2,700 for the 150-map robustness study.

Digitising the maps from Fig. 5 was done beforehand; the digitised maps are embedded in Part 4. With Internet access, Part 4.1 re-runs the digitiser on the figure and confirms it gives exactly the same maps.

## 0. Paper analysis and feasibility (assignment Section 3)

**Q1. Problem, objectives, contributions.**

*Problem.* Conventional A\* for global path planning on a grid is slow (many expanded nodes), adapts poorly to different maps (one fixed distance metric), and produces jagged paths with many sharp turns.

*Objective.* Shorter planning time, shorter and smoother paths.

*Contributions claimed by the paper:*
1. An **adaptive estimation function** f = g + ω(n)·h with ω = ln(1 + e^(d/D)).
2. A **hybrid distance model** h = (Euclidean + Manhattan)/2.
3. A **redundant point deletion** step based on collinearity and obstacle checks.

The paper reports, versus A\*: 69% less time, 2.66 m shorter paths, and 38.1% fewer turns ≥ 45°.

**Q2. Methodology.** A grid map with 1 m cells. A\* search with the weighted hybrid heuristic produces a raw grid path, and redundant point deletion turns it into the final path. The evaluation uses four metrics: time, length, expanded nodes, and corners ≥ 45°.

**Q3. Dataset.** There is no dataset, only three hand-made grid maps (30×30, 60×60, 100×100) with given start/goal points. **They are not publicly available**; they appear only as a figure (Fig. 5), which we digitised (Part 4).

**Q4. Tools and resources.**
- *Paper:* MATLAB R2023b on an Intel i5 laptop with 16 GB RAM.
- *Ours:* Python with NumPy, pandas, SciPy and Matplotlib, on CPU only. No GPU is needed.

**Q5. Feasibility.** Highly feasible: the method is deterministic and cheap, with no training. The main obstacles are the missing maps and the two baselines (Chen et al., Zhang et al.), which the paper does not describe.

**Q6. Enough detail?** Partly. The equations are clear, so the method and its trends can be reproduced. The exact numbers cannot, because the maps, several definitions and the test protocol are missing.

**Q7. Missing, unclear or hard to reproduce** (each becomes an assumption in Part 2):
- *Maps and motion model:* the obstacle maps; 4- vs 8-connected moves; whether diagonal moves may cut corners.
- *Definitions:* which distance d and D use; what "traversed nodes" means; the corner definition; how path length is measured.
- *Algorithm details:* the tie-breaking rule; the line-of-sight test and pruning order used for deletion.
- *Protocol:* the number of timing runs.
- *Baselines:* the Chen / Zhang methods; whether the ablation rows are isolated or cumulative.

**We also found** (see Parts 2.2, 5.1, 6 and 10):
- The heuristic **overestimates** the true 8-connected cost, so it is not admissible and optimality is lost (computed in Part 2.2). The hybrid term alone overestimates by up to 20.7%. The full adaptive estimate ω·h overestimates by up to **58.5%**, and V4 (no ½) by up to **3.17×**.
- The paper's own numbers contain contradictions. These are audited in Part 5.
- The paper's published Adaptive A\* node counts are *consistent with* the ½ of Eq. 7 being dropped in the full method (Part 10). This is a hypothesis; our main results follow the equations as printed.

## 1. Setup

In [ ]:
# Run configuration. QUICK = True gives a ~1-minute smoke run; use False for the reported numbers.
QUICK = False
TIMING_REPEATS = 3 if QUICK else 30     # timing repetitions per configuration (median reported)
TIMING_PAIRS = 30 if QUICK else 300     # interleaved A* / Adaptive A* pairs in the timing study
ROBUST_MAPS = 5 if QUICK else 50        # random maps per scene type in the robustness study

In [ ]:
%matplotlib inline
import gc, heapq, io, itertools, math, os, platform, statistics, time
from collections import deque
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import scipy
from scipy.stats import wilcoxon
from IPython.display import display, Markdown

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__}")
print("CPU:", platform.processor() or platform.machine(), "| cores:", __import__("os").cpu_count())

## 2. Implementation (written by us from the paper)

**Our implementation decisions.** These are needed wherever the paper is silent; each is applied to *every* planner alike.

| ID | Topic | Paper | Our choice |
|---|---|---|---|
| D1 | Maps | Figure only | Digitised from Fig. 5 (Part 4) |
| D2 | Motion | Not stated | 8-connected, cost 1 / √2, **diagonal corner cutting allowed**. Inferred: only this reproduces the paper's A\* / Dijkstra node counts |
| D3 | Baseline heuristic | "Traditional" | Euclidean (admissible, so our A\* baseline is optimal) |
| D4 | d, D in Eq. 6 | Not stated | Euclidean distances |
| D5 | Ties in OPEN | Not stated | Lower f, then lower h, then first-in-first-out (deterministic) |
| D6 | Line of sight | Not stated | **Bresenham** raster line, chosen because it reproduces the paper's pruned path lengths. This is calibration on pruning rows, i.e. on part of the proposed method, so the strict supercover test is reported too (Part 10.2) |
| D7 | Pruning order | Ambiguous | Anchor-based three-node window (alternative tested in Part 10) |
| D8 | Path length | Not stated | Euclidean length of the final polyline |
| D9 | Timing | One value at 0.01 s resolution | Median of 30 runs, plus a paired study (Part 7) |
| D10 | Chen / Zhang baselines | Not described | Compared on their published numbers only |
| D11 | Table 5 start (4,24) | Obstacle in the paper's own figure | That single cell is freed for that experiment only. Only this reproduces the paper's A\* exactly (moving the start instead gives 224–249 nodes vs 250; Part 6.3) |
| D12 | Full method | Eqs. 5–7 | Main results use the equations **exactly as printed**. The variant without the ½ of Eq. 7 ("V4") is reported separately, as a *hypothesis* about the authors' code, because it is consistent with the published node counts |

### 2.1 Environment model (paper Sec. 2.1)
The workspace is a grid of 1 m × 1 m cells; black cells are obstacles. The robot moves in **8 directions** (straight cost 1, diagonal cost √2).

**Assumption (not stated in the paper):** a diagonal move may not cut between two obstacles that touch at a corner.

In [ ]:
"""Occupancy-grid environment model (paper Sec. 2.1).

Cells are 1 m x 1 m squares. Internally a map is a boolean NumPy array
``occ[y, x]`` (True = obstacle) with 0-based indices. The paper uses 1-based
``(x, y)`` coordinates with the origin at the bottom-left, e.g. start ``(2, 2)``
on every map and goal ``(99, 99)`` on the 100 x 100 map; ``from_paper`` and
``to_paper`` convert between the two conventions.
"""


import math
from pathlib import Path

import numpy as np

SQRT2 = math.sqrt(2.0)

# 8-connected motion model (assumption M2): straight moves cost 1, diagonal sqrt(2).
STRAIGHT_MOVES = ((1, 0), (-1, 0), (0, 1), (0, -1))
DIAGONAL_MOVES = ((1, 1), (1, -1), (-1, 1), (-1, -1))

Cell = tuple[int, int]


def from_paper(x: int, y: int) -> Cell:
    """Convert a 1-based paper coordinate to a 0-based internal cell."""
    return (x - 1, y - 1)


def to_paper(cell: Cell) -> tuple[int, int]:
    """Convert a 0-based internal cell to a 1-based paper coordinate."""
    return (cell[0] + 1, cell[1] + 1)


class GridMap:
    """A 2-D occupancy grid with an 8-connected (or 4-connected) neighbourhood.

    Parameters
    ----------
    occ:
        Boolean array indexed ``[y, x]``; True marks an obstacle cell.
    connectivity:
        8 (default, as implied by the paper's 45-degree turn metric) or 4.
    allow_corner_cutting:
        If False (default, assumption M3) a diagonal move is only allowed when
        both orthogonally adjacent cells are free, so the robot never squeezes
        between two obstacles that touch at a corner.
    """

    def __init__(self, occ: np.ndarray, connectivity: int = 8,
                 allow_corner_cutting: bool = False, name: str = "map"):
        if connectivity not in (4, 8):
            raise ValueError("connectivity must be 4 or 8")
        self.occ = np.asarray(occ, dtype=bool)
        self.height, self.width = self.occ.shape
        self.connectivity = connectivity
        self.allow_corner_cutting = allow_corner_cutting
        self.name = name
        # Plain nested lists are much faster than NumPy for scalar look-ups
        # inside the Python search loop.
        self._occ_rows = self.occ.tolist()

    # ------------------------------------------------------------------ cells
    def in_bounds(self, x: int, y: int) -> bool:
        return 0 <= x < self.width and 0 <= y < self.height

    def is_free(self, x: int, y: int) -> bool:
        return 0 <= x < self.width and 0 <= y < self.height and not self._occ_rows[y][x]

    def neighbors(self, cell: Cell):
        """Yield ``(neighbour_cell, step_cost)`` pairs for a free cell."""
        x, y = cell
        rows = self._occ_rows
        w, h = self.width, self.height
        for dx, dy in STRAIGHT_MOVES:
            nx, ny = x + dx, y + dy
            if 0 <= nx < w and 0 <= ny < h and not rows[ny][nx]:
                yield (nx, ny), 1.0
        if self.connectivity == 4:
            return
        for dx, dy in DIAGONAL_MOVES:
            nx, ny = x + dx, y + dy
            if not (0 <= nx < w and 0 <= ny < h) or rows[ny][nx]:
                continue
            if not self.allow_corner_cutting and (rows[y][nx] or rows[ny][x]):
                continue
            yield (nx, ny), SQRT2

    # -------------------------------------------------------------------- I/O
    def save_csv(self, path: str | Path) -> None:
        """Save as CSV with the top row = largest y (so the file looks like the figure)."""
        np.savetxt(path, self.occ[::-1].astype(int), fmt="%d", delimiter=",")

    @classmethod
    def load_csv(cls, path: str | Path, **kwargs) -> "GridMap":
        arr = np.loadtxt(path, delimiter=",", dtype=int)[::-1]
        kwargs.setdefault("name", Path(path).stem)
        return cls(arr.astype(bool), **kwargs)

    def obstacle_ratio(self) -> float:
        return float(self.occ.mean())

    def __repr__(self) -> str:
        return (f"GridMap(name={self.name!r}, size={self.width}x{self.height}, "
                f"obstacles={self.obstacle_ratio():.1%}, connectivity={self.connectivity})")

### 2.2 Distance models, adaptive weight and hybrid heuristic (Sec. 2.2, 3.1, 3.2)

**Contribution 1 — adaptive estimation function (Eqs. 5–6):**
$$f(n) = g(n) + \omega(n)\,h(n), \qquad \omega(n) = \ln\!\left(1 + e^{d/D}\right)$$
where *d* is the distance from node *n* to the goal and *D* is the distance from start to goal.

**Contribution 2 — hybrid distance model (Eq. 7):**
$$h(n) = \frac{\text{Euclidean}(n, goal) + \text{Manhattan}(n, goal)}{2}$$

In [ ]:
"""Distance models and the adaptive heuristic weight (paper Sec. 2.2, 3.1, 3.2)."""


import math
from typing import Callable


DistanceFn = Callable[[Cell, Cell], float]
WeightFn = Callable[[Cell], float]


def euclidean(a: Cell, b: Cell) -> float:
    """Eq. (2): straight-line distance."""
    return math.hypot(a[0] - b[0], a[1] - b[1])


def manhattan(a: Cell, b: Cell) -> float:
    """Eq. (3): city-block distance."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def chebyshev(a: Cell, b: Cell) -> float:
    """Eq. (4): maximum axis difference."""
    return max(abs(a[0] - b[0]), abs(a[1] - b[1]))


def octile(a: Cell, b: Cell) -> float:
    """Exact 8-connected cost on an empty grid (not used by the paper; reference only)."""
    dx, dy = abs(a[0] - b[0]), abs(a[1] - b[1])
    return max(dx, dy) + (math.sqrt(2.0) - 1.0) * min(dx, dy)


def hybrid(a: Cell, b: Cell) -> float:
    """Eq. (7): arithmetic mean of the Euclidean and Manhattan distances."""
    dx, dy = abs(a[0] - b[0]), abs(a[1] - b[1])
    return 0.5 * (math.hypot(dx, dy) + dx + dy)


def hybrid_sum(a: Cell, b: Cell) -> float:
    """Euclidean + Manhattan WITHOUT the 1/2 of Eq. (7) (sensitivity variant "V4").

    Not the paper's equation. On the digitised paper maps, the full Adaptive A*
    built on this sum expands almost exactly the node counts published for the
    full method, while each component alone matches Eq. (6)/(7) as printed.
    """
    return 2.0 * hybrid(a, b)


def zero(a: Cell, b: Cell) -> float:
    """h = 0: A* degenerates into Dijkstra's algorithm."""
    return 0.0


DISTANCES: dict[str, DistanceFn] = {
    "euclidean": euclidean,
    "manhattan": manhattan,
    "chebyshev": chebyshev,
    "octile": octile,
    "hybrid": hybrid,
    "hybrid_sum": hybrid_sum,
    "zero": zero,
}


def adaptive_weight(start: Cell, goal: Cell, distance: DistanceFn = euclidean) -> WeightFn:
    """Eq. (6): omega(n) = ln(1 + exp(d / D)).

    ``d`` is the distance from node ``n`` to the goal and ``D`` the distance from
    the start to the goal, both measured with ``distance`` (assumption M4:
    Euclidean). omega = ln(1 + e) ~ 1.313 at the start and ln 2 ~ 0.693 at the goal.
    """
    big_d = distance(start, goal)
    if big_d == 0:
        return lambda n: 1.0

    def omega(n: Cell) -> float:
        return math.log1p(math.exp(distance(n, goal) / big_d))

    return omega


def constant_weight(value: float = 1.0) -> WeightFn:
    return lambda n: value

In [ ]:
# What does the adaptive weight look like?
w = adaptive_weight((0, 0), (10, 0))
print(f"omega at the start (d/D = 1): {w((0, 0)):.4f}   = ln(1+e)")
print(f"omega half way    (d/D = .5): {w((5, 0)):.4f}")
print(f"omega at the goal (d/D = 0): {w((10, 0)):.4f}   = ln 2")
# How much does the hybrid heuristic over-estimate the true 8-connected cost?
ratios = [hybrid((0, 0), (math.cos(t), math.sin(t))) / octile((0, 0), (math.cos(t), math.sin(t)))
          for t in np.linspace(0, math.pi / 2, 181)]
print(f"max hybrid / true cost = {max(ratios):.4f}  (at 45 degrees) -> the hybrid heuristic is NOT admissible")
# The search uses omega * h, and omega reaches ln(1+e) = 1.313 at the start (d = D):
w_max = math.log1p(math.e)
print(f"max omega * hybrid / true cost       = {w_max * max(ratios):.4f}  -> up to +{100 * (w_max * max(ratios) - 1):.1f}% (Eqs. 6-7 as printed)")
print(f"max omega * (E + M) / true cost (V4) = {2 * w_max * max(ratios):.4f}  -> about 3x: V4 is close to greedy best-first search")
print("(bounds for nodes no farther from the goal than the start; nodes behind the start get a larger omega)")

### 2.3 Search algorithms: A\* (Algorithm 1), Dijkstra, bidirectional A\*
- **A\*:** an OPEN list (binary heap) and a CLOSED list (set). Nodes on CLOSED are never re-opened, and the search stops when the goal is popped, exactly as in the paper's Algorithm 1. The optional `weight` argument turns plain A\* into the paper's adaptive A\*.
- **Dijkstra:** A\* with h = 0.
- **Bidirectional A\*:** a *generic* version, because the paper does not describe its ref. [30]. A forward and a backward search alternate one expansion each and **stop at the first meeting**, i.e. when one side expands a node the other side has already closed. This rule is **not guaranteed to give the shortest path**; an optimal version would continue until min f_forward + min f_backward ≥ the best meeting cost. The paper's bidirectional paths are 1.6–2.2 m longer than its A\*, which fits such a rule.

In [ ]:
"""Grid search algorithms: (weighted) A* per Algorithm 1, Dijkstra and bidirectional A*.

All searches are written from scratch on top of ``heapq``. Counting convention
(assumption M6): ``expanded`` is the number of nodes moved to the CLOSED list,
``generated`` is the number of distinct nodes ever placed on the OPEN list.
"""


import heapq
import itertools
from dataclasses import dataclass, field


@dataclass
class SearchResult:
    path: list[Cell]            # start ... goal; empty if no path exists
    cost: float                 # accumulated g-cost of the path (1 / sqrt(2) steps)
    expanded: int               # |CLOSED|
    generated: int              # distinct nodes ever opened
    closed: set[Cell] = field(default_factory=set, repr=False)

    @property
    def found(self) -> bool:
        return bool(self.path)


def _reconstruct(parent: dict[Cell, Cell | None], node: Cell) -> list[Cell]:
    path = [node]
    while parent[node] is not None:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path


def astar(grid: GridMap, start: Cell, goal: Cell,
          heuristic: DistanceFn = euclidean, weight: WeightFn | None = None) -> SearchResult:
    """Best-first search with ``f(n) = g(n) + w(n) * h(n)`` (Eq. 1 / Eq. 5).

    Follows the paper's Algorithm 1: a node is finalised when popped from OPEN,
    CLOSED nodes are never re-opened, and the search stops when the goal is popped.
    Ties in ``f`` are broken by smaller ``h`` and then by insertion order
    (assumption M8), which makes every run deterministic.
    """
    if not (grid.is_free(*start) and grid.is_free(*goal)):
        return SearchResult([], float("inf"), 0, 0)

    h_cache: dict[Cell, float] = {}

    def h_term(n: Cell) -> float:
        val = h_cache.get(n)
        if val is None:
            hn = heuristic(n, goal)
            val = hn * weight(n) if weight is not None else hn
            h_cache[n] = val
        return val

    counter = itertools.count()
    g: dict[Cell, float] = {start: 0.0}
    parent: dict[Cell, Cell | None] = {start: None}
    hs = h_term(start)
    open_heap = [(hs, hs, next(counter), start)]
    closed: set[Cell] = set()

    while open_heap:
        _, _, _, current = heapq.heappop(open_heap)
        if current in closed:          # stale heap entry (lazy deletion)
            continue
        if current == goal:
            closed.add(current)
            return SearchResult(_reconstruct(parent, current), g[current],
                                len(closed), len(g), closed)
        closed.add(current)
        g_cur = g[current]
        for nb, step in grid.neighbors(current):
            if nb in closed:
                continue
            g_new = g_cur + step
            if g_new < g.get(nb, float("inf")):
                g[nb] = g_new
                parent[nb] = current
                hn = h_term(nb)
                heapq.heappush(open_heap, (g_new + hn, hn, next(counter), nb))

    return SearchResult([], float("inf"), len(closed), len(g), closed)


def dijkstra(grid: GridMap, start: Cell, goal: Cell) -> SearchResult:
    """Dijkstra's algorithm = A* with h(n) = 0."""
    return astar(grid, start, goal, heuristic=zero)


def bidirectional_astar(grid: GridMap, start: Cell, goal: Cell,
                        heuristic: DistanceFn = euclidean) -> SearchResult:
    """Bidirectional A* (assumption M14).

    A forward search (towards ``goal``) and a backward search (towards ``start``)
    alternate one expansion each. The search stops as soon as a node expanded by
    one side has already been expanded by the other, and the two half-paths are
    joined there. Stopping at the first meeting node (rather than continuing
    until the optimal-termination bound is met) is the textbook simple variant;
    it is fast but may return a slightly longer path, matching the paper's
    observation that BA* paths are longer.

    An optimal bidirectional search would keep the best meeting cost mu seen so
    far and stop only when min f_forward + min f_backward >= mu (or a similar
    bound). The paper does not describe the method of its ref. [30], so this is
    a standard generic version, not a re-implementation of [30].
    """
    if not (grid.is_free(*start) and grid.is_free(*goal)):
        return SearchResult([], float("inf"), 0, 0)
    if start == goal:
        return SearchResult([start], 0.0, 1, 1, {start})

    counter = itertools.count()
    sides = []
    for origin, target in ((start, goal), (goal, start)):
        h0 = heuristic(origin, target)
        sides.append({
            "target": target,
            "g": {origin: 0.0},
            "parent": {origin: None},
            "open": [(h0, h0, next(counter), origin)],
            "closed": set(),
        })

    turn = 0
    while sides[0]["open"] and sides[1]["open"]:
        me, other = sides[turn], sides[1 - turn]
        _, _, _, current = heapq.heappop(me["open"])
        if current in me["closed"]:
            continue
        me["closed"].add(current)
        if current in other["closed"]:
            fwd, bwd = sides[0], sides[1]
            first = _reconstruct(fwd["parent"], current)
            second = _reconstruct(bwd["parent"], current)[::-1][1:]
            closed = fwd["closed"] | bwd["closed"]
            cost = fwd["g"][current] + bwd["g"][current]
            generated = len(set(fwd["g"]) | set(bwd["g"]))
            return SearchResult(first + second, cost,
                                len(fwd["closed"]) + len(bwd["closed"]), generated, closed)
        g_cur = me["g"][current]
        for nb, step in grid.neighbors(current):
            if nb in me["closed"]:
                continue
            g_new = g_cur + step
            if g_new < me["g"].get(nb, float("inf")):
                me["g"][nb] = g_new
                me["parent"][nb] = current
                hn = heuristic(nb, me["target"])
                heapq.heappush(me["open"], (g_new + hn, hn, next(counter), nb))
        turn = 1 - turn

    expanded = len(sides[0]["closed"]) + len(sides[1]["closed"])
    return SearchResult([], float("inf"), expanded, len(set(sides[0]["g"]) | set(sides[1]["g"])))

### 2.4 Redundant point deletion (Contribution 3, Sec. 3.3, Fig. 4)
A three-node window (P*i*, P*i*+1, P*i*+2) moves along the raw path:
- **Step 2:** if the three nodes are collinear, delete the middle one.
- **Step 3:** otherwise, connect P*i* to P*i*+2. If that straight line touches no obstacle cell (*line-of-sight* test), delete the middle node; if it does, keep it.

The paper's Fig. 4 keeps node A fixed while B is deleted and C advances, so we implement an **anchor-based** window. The line-of-sight test can be the strict "supercover" test or a Bresenham raster line; the paper's numbers indicate Bresenham (Part 4).

In [ ]:
"""Redundant point deletion strategy (paper Sec. 3.3, Fig. 4).

Path nodes are cell centres. Two line-of-sight tests are available (assumption M9):

* ``"supercover"`` (default, conservative): a segment is free only if it touches
  no obstacle cell; cells are closed squares ``[x-0.5, x+0.5] x [y-0.5, y+0.5]``,
  so a segment that merely grazes an obstacle corner is rejected.
* ``"bresenham"``: only the cells of the Bresenham raster line must be free.
  This is the usual MATLAB-style test; it lets a segment clip obstacle corners.
  On the digitised paper maps it reproduces the paper's path lengths closely.
"""


import math


_EPS = 1e-9


def segment_cells(a: Cell, b: Cell) -> list[Cell]:
    """All grid cells whose closed square intersects the segment a-b (exact supercover)."""
    (x0, y0), (x1, y1) = a, b
    cells: list[Cell] = []
    if x0 == x1:
        for y in range(min(y0, y1), max(y0, y1) + 1):
            cells.append((x0, y))
        return cells
    if x0 > x1:
        (x0, y0), (x1, y1) = (x1, y1), (x0, y0)
    slope = (y1 - y0) / (x1 - x0)
    for cx in range(x0, x1 + 1):
        # Portion of the segment lying inside column cx (|x - cx| <= 0.5).
        xa = max(cx - 0.5, x0)
        xb = min(cx + 0.5, x1)
        ya = y0 + slope * (xa - x0)
        yb = y0 + slope * (xb - x0)
        lo, hi = min(ya, yb), max(ya, yb)
        # Rows r whose interval [r - 0.5, r + 0.5] meets [lo, hi].
        r_lo = math.ceil(lo - 0.5 - _EPS)
        r_hi = math.floor(hi + 0.5 + _EPS)
        for cy in range(r_lo, r_hi + 1):
            cells.append((cx, cy))
    # Column boundaries are closed too: a segment passing exactly through the
    # boundary x = cx + 0.5 already enumerates both columns above.
    return cells


def bresenham_cells(a: Cell, b: Cell) -> list[Cell]:
    """Cells of the Bresenham raster line from a to b (inclusive)."""
    (x0, y0), (x1, y1) = a, b
    dx, dy = abs(x1 - x0), -abs(y1 - y0)
    sx, sy = (1 if x0 < x1 else -1), (1 if y0 < y1 else -1)
    err = dx + dy
    cells = []
    while True:
        cells.append((x0, y0))
        if (x0, y0) == (x1, y1):
            return cells
        e2 = 2 * err
        if e2 >= dy:
            err += dy
            x0 += sx
        if e2 <= dx:
            err += dx
            y0 += sy


LOS_METHODS = {"supercover": segment_cells, "bresenham": bresenham_cells}


def line_of_sight(grid: GridMap, a: Cell, b: Cell, method: str = "supercover") -> bool:
    """True if the straight segment between cell centres a and b is obstacle-free."""
    for x, y in LOS_METHODS[method](a, b):
        if not grid.is_free(x, y):
            return False
    return True


def is_collinear(p: Cell, q: Cell, r: Cell) -> bool:
    """Collinearity test via the 2-D cross product (Step 2)."""
    return (q[0] - p[0]) * (r[1] - p[1]) - (q[1] - p[1]) * (r[0] - p[0]) == 0


def delete_redundant_points(grid: GridMap, path: list[Cell], mode: str = "anchor",
                            los: str = "supercover") -> list[Cell]:
    """Remove redundant nodes from a grid path (Steps 1-4 of Sec. 3.3).

    Walk the path with a three-node window (P_i, P_i+1, P_i+2) anchored at P_i:
      * Step 2 - if the three nodes are collinear and the segment is free,
        delete the middle node;
      * Step 3 - otherwise connect P_i to P_i+2 and delete the middle node if
        that line does not cross an obstacle, else keep it and advance the anchor.
    After a deletion the anchor stays put and the window slides to the next
    node (assumption M10), so the anchor keeps "pulling the string" until line
    of sight is lost. Step 4 joins the kept nodes into the new path.

    ``mode="single-pass"`` is the alternative, more literal reading used as a
    sensitivity check: the window is never re-anchored, i.e. after deleting the
    middle node the window jumps to the next triple (P_i+2, P_i+3, P_i+4), so
    each node is examined once and at most every other node can be removed.
    """
    if mode not in ("anchor", "single-pass"):
        raise ValueError("mode must be 'anchor' or 'single-pass'")
    if len(path) <= 2:
        return list(path)
    kept = [path[0]]
    anchor = path[0]
    i = 1
    while i < len(path) - 1:
        middle, nxt = path[i], path[i + 1]
        # Step 2: with the supercover test a collinear triple is free by construction
        # (anchor->middle was already verified and middle->nxt is a free grid step).
        # Step 3: otherwise the shortcut anchor->nxt needs an explicit line-of-sight test.
        removable = (los == "supercover" and is_collinear(anchor, middle, nxt)) \
            or line_of_sight(grid, anchor, nxt, los)
        if not removable:
            kept.append(middle)                    # Step 3: shortcut blocked, keep node
            anchor = middle
            i += 1
        elif mode == "anchor":
            i += 1                                 # keep the anchor, test the next node
        else:
            if i + 1 < len(path) - 1:
                kept.append(nxt)                   # window jumps past the deleted node
            anchor = nxt
            i += 2
    kept.append(path[-1])
    return kept

### 2.5 Evaluation metrics (as in Tables 1–5)
Path length (m), expanded nodes (size of CLOSED), number of corners with heading change ≥ 45°, and planning time.

In [ ]:
"""Evaluation metrics used in the paper's Tables 1-5."""


import math


def path_length(path: list[Cell], cell_size: float = 1.0) -> float:
    """Euclidean polyline length in metres (assumption M16)."""
    return cell_size * sum(math.dist(path[i], path[i + 1]) for i in range(len(path) - 1))


def turn_angles(path: list[Cell]) -> list[float]:
    """Absolute heading change (degrees) at every interior node."""
    angles = []
    for i in range(1, len(path) - 1):
        ax, ay = path[i][0] - path[i - 1][0], path[i][1] - path[i - 1][1]
        bx, by = path[i + 1][0] - path[i][0], path[i + 1][1] - path[i][1]
        cross = ax * by - ay * bx
        dot = ax * bx + ay * by
        angles.append(abs(math.degrees(math.atan2(cross, dot))))
    return angles


def count_sharp_turns(path: list[Cell], threshold_deg: float = 45.0) -> int:
    """Number of corners whose heading change is >= threshold (assumption M7)."""
    return sum(1 for a in turn_angles(path) if a >= threshold_deg - 1e-9)

### 2.6 Planners: every combination needed for the experiments
`A*` is the baseline. The paper's three improvements can each be switched on alone (the ablation of Table 1) or all together (**Adaptive A\***).

In [ ]:
"""Named planners used in the experiments, built from the three switches of the paper.

Every planner returns a :class:`PlanResult` with the raw grid path, the final
path (after redundant point deletion when enabled) and the four paper metrics.

=====================  ===============  ==============  =============  =======
Planner                heuristic h(n)   weight w(n)     pruning        paper
=====================  ===============  ==============  =============  =======
A*                     Euclidean        1               no             Tab 1-5
Dijkstra               0                -               no             Tab 2-4
Bidirectional A*       Euclidean        1               no             Tab 2-4
AEF (isolated)         Euclidean        Eq. (6)         no             Tab 1
DMO (isolated)         hybrid Eq. (7)   1               no             Tab 1
RPD (isolated)         Euclidean        1               yes            Tab 1
AEF+DMO (cumulative)   hybrid           Eq. (6)         no             extra
Adaptive A*            hybrid           Eq. (6)         yes            Tab 1-5
A*+Octile+RPD          octile           1               yes            extra
... (single-pass)      as above         as above        single-pass    extra
... (V4: E+M w/o 1/2)  Euclid+Manhattan Eq. (6)         yes            extra
=====================  ===============  ==============  =============  =======
"""


import gc
import statistics
import time
from dataclasses import dataclass
from typing import Callable


@dataclass
class PlanResult:
    planner: str
    raw_path: list[Cell]
    path: list[Cell]
    expanded: int
    generated: int
    raw_cost: float
    length: float
    corners: int
    closed: set[Cell]

    @property
    def found(self) -> bool:
        return bool(self.path)


@dataclass(frozen=True)
class PlannerSpec:
    name: str
    search: str = "astar"          # astar | dijkstra | bidirectional
    heuristic: str = "euclidean"   # euclidean | hybrid | octile
    adaptive: bool = False         # use the adaptive weight of Eq. (6)
    prune: bool = False            # apply redundant point deletion (Sec. 3.3)
    prune_mode: str = "anchor"     # anchor | single-pass (sensitivity, assumption M10)
    los: str = "supercover"        # supercover | bresenham line-of-sight test (assumption M9)

    def search_only(self, grid: GridMap, start: Cell, goal: Cell) -> SearchResult:
        h = {"euclidean": euclidean, "hybrid": hybrid, "hybrid_sum": hybrid_sum,
             "octile": octile}[self.heuristic]
        if self.search == "dijkstra":
            return dijkstra(grid, start, goal)
        if self.search == "bidirectional":
            return bidirectional_astar(grid, start, goal, heuristic=h)
        weight = adaptive_weight(start, goal) if self.adaptive else None
        return astar(grid, start, goal, heuristic=h, weight=weight)

    def plan(self, grid: GridMap, start: Cell, goal: Cell) -> PlanResult:
        res = self.search_only(grid, start, goal)
        final = (delete_redundant_points(grid, res.path, self.prune_mode, self.los)
                 if (self.prune and res.found) else res.path)
        return PlanResult(
            planner=self.name, raw_path=res.path, path=final,
            expanded=res.expanded, generated=res.generated, raw_cost=res.cost,
            length=path_length(final) if final else float("nan"),
            corners=count_sharp_turns(final) if final else -1,
            closed=res.closed,
        )


PLANNERS: dict[str, PlannerSpec] = {p.name: p for p in [
    PlannerSpec("A*"),
    PlannerSpec("Dijkstra", search="dijkstra"),
    PlannerSpec("Bidirectional A*", search="bidirectional"),
    PlannerSpec("Adaptive estimation function", adaptive=True),
    PlannerSpec("Distance model optimization", heuristic="hybrid"),
    PlannerSpec("Redundant point deletion", prune=True),
    PlannerSpec("AEF+DMO (no pruning)", heuristic="hybrid", adaptive=True),
    PlannerSpec("Adaptive A*", heuristic="hybrid", adaptive=True, prune=True),
    PlannerSpec("A*+Octile+RPD", heuristic="octile", prune=True),
    PlannerSpec("Redundant point deletion (single-pass)", prune=True, prune_mode="single-pass"),
    PlannerSpec("Adaptive A* (single-pass pruning)", heuristic="hybrid", adaptive=True,
                prune=True, prune_mode="single-pass"),
    PlannerSpec("Adaptive A* (V4: E+M without 1/2)", heuristic="hybrid_sum", adaptive=True, prune=True),
]}


def time_planner(spec: PlannerSpec, grid: GridMap, start: Cell, goal: Cell,
                 repeats: int = 30, warmup: int = 2) -> dict[str, float]:
    """Wall-clock time of search + pruning (assumption M11/M12).

    Returns median, inter-quartile range, mean and std over ``repeats`` runs,
    measured with ``time.perf_counter`` and the garbage collector disabled.
    """
    run: Callable[[], PlanResult] = lambda: spec.plan(grid, start, goal)
    for _ in range(warmup):
        run()
    samples = []
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        for _ in range(repeats):
            t0 = time.perf_counter()
            run()
            samples.append(time.perf_counter() - t0)
    finally:
        if gc_was_enabled:
            gc.enable()
    q = statistics.quantiles(samples, n=4) if len(samples) >= 2 else [samples[0]] * 3
    return {
        "time_median": statistics.median(samples),
        "time_iqr": q[2] - q[0],
        "time_mean": statistics.fmean(samples),
        "time_std": statistics.pstdev(samples),
        "repeats": repeats,
    }

## 3. Sanity checks
These tests confirm the implementation behaves correctly before any experiment is run.

In [ ]:
def check(name, condition):
    print(("PASS " if condition else "FAIL ") + name)
    assert condition, name

e = GridMap(np.zeros((10, 10), dtype=bool))
check("A* on an empty grid finds the exact diagonal cost 9*sqrt(2)",
      abs(astar(e, (0, 0), (9, 9)).cost - 9 * math.sqrt(2)) < 1e-9)
occ = np.zeros((3, 3), dtype=bool); occ[0, 1] = occ[1, 0] = True
check("no diagonal corner cutting between touching obstacles",
      (1, 1) not in [n for n, _ in GridMap(occ).neighbors((0, 0))])
occ = np.zeros((5, 5), dtype=bool); occ[2, 2] = True; g5 = GridMap(occ)
check("line of sight is blocked through an obstacle", not line_of_sight(g5, (0, 0), (4, 4)))
check("line of sight is free along an empty row", line_of_sight(g5, (0, 0), (4, 0)))
check("pruning straightens a staircase into one segment",
      delete_redundant_points(e, [(0, 0), (1, 0), (2, 0), (3, 1), (4, 2), (5, 3)]) == [(0, 0), (5, 3)])
check("a 45-degree turn is counted as a corner", count_sharp_turns([(0, 0), (1, 0), (2, 1)]) == 1)
check("adaptive weight = ln(1+e) at start and ln 2 at goal",
      abs(adaptive_weight((0, 0), (10, 0))((0, 0)) - math.log(1 + math.e)) < 1e-12
      and abs(adaptive_weight((0, 0), (10, 0))((10, 0)) - math.log(2)) < 1e-12)

## 4. Maps: digitised from the paper's Fig. 5 (our reconstruction)

The paper publishes its three maps **only as a picture** (Fig. 5). We converted that picture into occupancy grids ourselves:

| Map | Pixels per cell | How we digitised it |
|---|---|---|
| 30×30 (irregular) | ≈ 4.9 | Each cell classified by the mean blackness around its centre. Red start / blue goal cells are not black, so they stay free |
| 60×60 (maze) | ≈ 2.45 | Cell pitch fixed exactly by the two black border walls; each cell centre sampled on a 10× upsampled image |
| 100×100 (array) | ≈ 1.49 | Too fine for per-cell sampling, so each dark block (all 5×5 cells) is detected as a whole. Block groups were then refined by ±1 cell, using **only baseline** statistics |

**Validation (next cells).** On these maps our *baseline* A\* and Dijkstra reproduce the paper's published numbers almost exactly. For example: A\* 130 vs 130 nodes, 715 vs 716, and 2188 vs 2191; Dijkstra 633 vs 632, 1741 vs 1740, and 7521 vs 7520. The Table 5 pair (4,24)→(29,2) gives 38.21 m / 250 nodes / 14 corners, identical to the paper.

*Honest note:* maps 1 and 2 were not tuned, so their agreement is a real check. Map 3's block positions and the corner-cutting setting were **chosen to fit these baseline numbers** (calibration, Part 10.2), so for them the agreement is partly built in.

This also **resolves two settings the paper never states:**
- Diagonal moves may cut obstacle corners. Without corner cutting the node counts do not match.
- Pruning uses a Bresenham line of sight. This reproduces the paper's pruned path lengths.

The random maps of Part 9 still come from our procedural generators (code below).

In [ ]:
"""Procedural generators for the three scene types of the paper (Sec. 4.1, Fig. 5).

The original obstacle matrices are not published (missing item M1), so the
experiments use *proxy maps*: maps of the same size and scene type whose
parameters are calibrated (``archive/calibrate_maps.py``) so that Dijkstra and
A* reproduce the paper's published search statistics as closely as possible.

* ``irregular``  - map1, 30 x 30, irregular outdoor scene (random blobs);
* ``maze``       - map2, 60 x 60, structured indoor scene (maze + corridors);
* ``array``      - map3, 100 x 100, array-like scene (PV panel rows / racks).

All generators take an explicit ``numpy.random.Generator`` so every map is
reproducible from its seed.
"""


from collections import deque

import numpy as np


# Start/goal pairs used in the paper (1-based paper coordinates).
PAPER_POINTS: dict[str, list[tuple[int, int]]] = {
    "map1": [(2, 2), (28, 28), (4, 24), (29, 2)],
    "map2": [(2, 2), (58, 58), (4, 54), (59, 2)],
    "map3": [(2, 2), (99, 99)],
}
MAP_SIZES = {"map1": 30, "map2": 60, "map3": 100}

# The two map sets used in the experiments.
#  * "paper": the paper's own maps, digitised from its Fig. 5 (scripts/digitize_figure5.py).
#    Diagonal corner cutting is allowed and pruning uses a Bresenham line of sight: with
#    these settings A* and Dijkstra reproduce the paper's published node counts.
#  * "proxy": procedurally generated maps calibrated to the paper's statistics
#    (archive/calibrate_maps.py), with the conservative settings (no corner cutting,
#    supercover line of sight). Used before the figure was available, and for the
#    random-map robustness study.
MAP_SETS = {
    "paper": {"files": {"30x30": "paper_map1", "60x60": "paper_map2", "100x100": "paper_map3"},
              "corner_cutting": True, "los": "bresenham"},
    "proxy": {"files": {"30x30": "map1", "60x60": "map2", "100x100": "map3"},
              "corner_cutting": False, "los": "supercover"},
}


def load_map_set(name: str, maps_dir) -> dict:
    """Load the three maps of a map set as ``{"30x30": GridMap, ...}``."""
    from pathlib import Path
    cfg = MAP_SETS[name]
    return {size: GridMap.load_csv(Path(maps_dir) / f"{stem}.csv", allow_corner_cutting=cfg["corner_cutting"])
            for size, stem in cfg["files"].items()}
MAP_KINDS = {"map1": "irregular", "map2": "maze", "map3": "array"}


# --------------------------------------------------------------- generators
def irregular(n: int, rng: np.random.Generator, n_blobs: int = 18,
              blob_cells: tuple[int, int] = (6, 30)) -> np.ndarray:
    """Irregular outdoor obstacles: blobs grown by random cell accretion."""
    occ = np.zeros((n, n), dtype=bool)
    for _ in range(n_blobs):
        size = int(rng.integers(blob_cells[0], blob_cells[1] + 1))
        blob = [(int(rng.integers(0, n)), int(rng.integers(0, n)))]
        occ[blob[0][1], blob[0][0]] = True
        attempts = 0
        while len(blob) < size and attempts < 50 * size:   # a boxed-in blob just stops growing
            attempts += 1
            x, y = blob[int(rng.integers(0, len(blob)))]
            dx, dy = ((1, 0), (-1, 0), (0, 1), (0, -1))[int(rng.integers(0, 4))]
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n and not occ[ny, nx]:
                occ[ny, nx] = True
                blob.append((nx, ny))
    return occ


def maze(n: int, rng: np.random.Generator, corridor: int = 3, wall: int = 2,
         loop_prob: float = 0.15) -> np.ndarray:
    """Structured indoor scene: a randomised-DFS maze with corridors of width
    ``corridor`` and walls of thickness ``wall``; a fraction ``loop_prob`` of the
    remaining internal walls is opened to create alternative routes."""
    pitch = corridor + wall
    m = max(2, (n - wall) // pitch)                    # coarse cells per side
    occ = np.ones((n, n), dtype=bool)
    offset = (n - (m * pitch + wall)) // 2

    def room_origin(i: int, j: int) -> tuple[int, int]:
        return offset + wall + i * pitch, offset + wall + j * pitch

    for i in range(m):
        for j in range(m):
            x0, y0 = room_origin(i, j)
            occ[y0:y0 + corridor, x0:x0 + corridor] = False

    def open_between(a: tuple[int, int], b: tuple[int, int]) -> None:
        (i0, j0), (i1, j1) = a, b
        x0, y0 = room_origin(min(i0, i1), min(j0, j1))
        if i0 != i1:   # horizontal neighbours: knock out the vertical wall
            occ[y0:y0 + corridor, x0 + corridor:x0 + pitch] = False
        else:
            occ[y0 + corridor:y0 + pitch, x0:x0 + corridor] = False

    visited = np.zeros((m, m), dtype=bool)
    stack = [(0, 0)]
    visited[0, 0] = True
    walls_left = []
    while stack:
        i, j = stack[-1]
        nbrs = [(i + di, j + dj) for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1))
                if 0 <= i + di < m and 0 <= j + dj < m and not visited[i + di, j + dj]]
        if not nbrs:
            stack.pop()
            continue
        k = nbrs[int(rng.integers(0, len(nbrs)))]
        open_between((i, j), k)
        visited[k] = True
        stack.append(k)
    for i in range(m):
        for j in range(m):
            for di, dj in ((1, 0), (0, 1)):
                if i + di < m and j + dj < m:
                    walls_left.append(((i, j), (i + di, j + dj)))
    for a, b in walls_left:
        if rng.random() < loop_prob:
            open_between(a, b)
    return occ


def array_field(n: int, rng: np.random.Generator, panel_h: int = 2, aisle: int = 3,
                panel_len: tuple[int, int] = (8, 16), gap: tuple[int, int] = (1, 3),
                missing_prob: float = 0.08, clutter: int = 20) -> np.ndarray:
    """Array-like scene: staggered rows of rectangular panels (PV strings,
    warehouse racks) separated by aisles, with random gaps, missing panels and
    a little random clutter."""
    occ = np.zeros((n, n), dtype=bool)
    y = aisle
    while y + panel_h < n - 1:
        x = int(rng.integers(0, panel_len[1]))
        while x < n:
            length = int(rng.integers(panel_len[0], panel_len[1] + 1))
            if rng.random() >= missing_prob:
                occ[y:y + panel_h, x:min(n, x + length)] = True
            x += length + int(rng.integers(gap[0], gap[1] + 1))
        y += panel_h + aisle
    for _ in range(clutter):
        cx, cy = int(rng.integers(0, n)), int(rng.integers(0, n))
        occ[cy:cy + int(rng.integers(1, 3)), cx:cx + int(rng.integers(1, 3))] = True
    return occ


GENERATORS = {"irregular": irregular, "maze": maze, "array": array_field}


# ----------------------------------------------------------------- helpers
def clear_points(occ: np.ndarray, cells: list[Cell], radius: int = 1) -> None:
    """Free the given cells and their (2r+1)^2 neighbourhood in place."""
    h, w = occ.shape
    for x, y in cells:
        occ[max(0, y - radius):min(h, y + radius + 1), max(0, x - radius):min(w, x + radius + 1)] = False


def connected(grid: GridMap, cells: list[Cell]) -> bool:
    """True if all cells lie in one 8-connected free component (BFS)."""
    if not all(grid.is_free(*c) for c in cells):
        return False
    seen = {cells[0]}
    queue = deque([cells[0]])
    while queue:
        cur = queue.popleft()
        for nb, _ in grid.neighbors(cur):
            if nb not in seen:
                seen.add(nb)
                queue.append(nb)
    return all(c in seen for c in cells)


def largest_component(grid: GridMap) -> set[Cell]:
    """Cells of the largest 8-connected free component."""
    unseen = {(x, y) for y in range(grid.height) for x in range(grid.width) if grid.is_free(x, y)}
    best: set[Cell] = set()
    while unseen:
        seed = unseen.pop()
        comp = {seed}
        queue = deque([seed])
        while queue:
            for nb, _ in grid.neighbors(queue.popleft()):
                if nb in unseen:
                    unseen.discard(nb)
                    comp.add(nb)
                    queue.append(nb)
        if len(comp) > len(best):
            best = comp
    return best


def carve_access(occ: np.ndarray, cells: list[Cell]) -> None:
    """Connect each required cell to the main free region with an L-shaped,
    1-cell-wide corridor (needed when a start/goal falls inside a wall)."""
    comp = largest_component(GridMap(occ))
    for x, y in cells:
        if (x, y) in comp:
            continue
        tx, ty = min(comp, key=lambda c: (c[0] - x) ** 2 + (c[1] - y) ** 2)
        occ[y, min(x, tx):max(x, tx) + 1] = False
        occ[min(y, ty):max(y, ty) + 1, tx] = False
        comp = largest_component(GridMap(occ))


def make_map(map_id: str, seed: int, **params) -> GridMap | None:
    """Generate a proxy map for ``map_id`` ("map1" | "map2" | "map3").

    The paper's start/goal cells are freed and, if necessary, linked to the
    main free region. Returns ``None`` if they are still not mutually reachable.
    """
    n = MAP_SIZES[map_id]
    rng = np.random.default_rng(seed)
    occ = GENERATORS[MAP_KINDS[map_id]](n, rng, **params)
    cells = [from_paper(*p) for p in PAPER_POINTS[map_id]]
    clear_points(occ, cells, radius=1)
    carve_access(occ, cells)
    grid = GridMap(occ, name=f"{map_id}_seed{seed}")
    return grid if connected(grid, cells) else None

In [ ]:
# ---- Our digitisation of Fig. 5 (scripts/digitize_figure5.py). '#' = obstacle; first row = top of the figure.
MAP1_ROWS = [
    "##.........##.#...............",
    "#####......####......####.....",
    "#####.......#........####.....",
    "#######........###...####.....",
    ".####..........#.....####.....",
    "...#####.......####...........",
    "...#.......##...#.............",
    ".#......#####.....#.......###.",
    "##.....###..#.....#........##.",
    "##....###...#....####.......#.",
    "#......#....#.....#.....#.....",
    "............####..#.....#.....",
    ".##..##.............#...#.....",
    "..#...........##....#...#.....",
    "..#........#####........#.....",
    "..#.#......#......#.....#.....",
    "..#.####...#.....#####..#..###",
    ".......##..####...##....#..#..",
    "##.#####......#...####..#.....",
    ".#..#.........#...#.#.........",
    "....#.........#..##.#.........",
    ".......##.....#............#..",
    "##.....##.....#......##....###",
    "#####..###....#......###......",
    "##..................###....##.",
    "##............##..#.####...##.",
    "......##.......#.....#.....###",
    "..##...#######.......#........",
    "..##................##...##...",
    "..##...................###...."
]

MAP2_ROWS = [
    "########################......##############################",
    "#.............#........................#.........#.........#",
    "#.............#........................#.........#.........#",
    "#.............#........................#.........#.........#",
    "#.............#........................#.........#.........#",
    "#........###########....############...######..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#........#......#...#....#..............#",
    "#........#.........#....#...#......#...#....#....###########",
    "#........#.........#....#...#......#...#....#....#.........#",
    "#........#.........#....#...#......#...#....#....#.........#",
    "#........#....#....#....#...#......#...#....#....#.........#",
    "#.............#.........#...#......#...#....#....#.........#",
    "#.............#.........#...#......#...#....#....#.........#",
    "#.............#.........#...#......#...#....#....#.........#",
    "#.............#.........#...#......#...#....#....#.........#",
    "#........###########....#####......#...#....#....#.........#",
    "#.............#.............#..............................#",
    "#.............#.............#..............................#",
    "#.............#.............#..............................#",
    "#.............#.............#..............................#",
    "#........####################......#...###########.........#",
    "#........#....#....#....#..........#........#....#.........#",
    "#........#....#....#....#..........#........#....#.........#",
    "#........#....#....#....#..........#........#....#.........#",
    "#........#....#....#....#..........#........#....#.........#",
    "##########....#....#....#...#################....#.........#",
    "#...........................#..........#....#..............#",
    "#...........................#..........#....#..............#",
    "#...........................#..........#....#..............#",
    "#...........................#..........#....#..............#",
    "#........###########....############...#....#....###########",
    "#........#....#.........#..........#.......................#",
    "#........#....#.........#..........#.......................#",
    "#........#....#.........#..........#.......................#",
    "#........#....#.........#..........#.......................#",
    "#........#....###############......#...######....#.........#",
    "#.............#.............#......#...#.........#.........#",
    "#.............#.............#......#...#.........#.........#",
    "#.............#.............#......#...#.........#.........#",
    "#.............#.............#......#...#.........#.........#",
    "#........#....#....######...#......##########....#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...#..........#.........#.........#",
    "#........#....#....#....#...########...###########.........#",
    "#........#..............#..........#.......................#",
    "#........#..............#..........#.......................#",
    "#........#..............#..........#.......................#",
    "#############################......#########################"
]

# map3 is a regular lattice of 5x5-cell blocks; lower-left corners in paper coordinates
MAP3_XS = [5, 13, 21, 35, 43, 51, 65, 73, 81, 92]
MAP3_YS = [6, 14, 22, 36, 44, 52, 66, 74, 82, 93]
MAP3_MISSING = {(92, 93)}

CORNER_CUTTING = True    # inferred: with it, A* and Dijkstra reproduce the paper's node counts
LOS = "bresenham"        # inferred: reproduces the paper's pruned path lengths

def grid_from_rows(rows, name):
    occ = np.array([[ch == "#" for ch in row] for row in rows])[::-1]     # internal grids are bottom-up
    return GridMap(occ, allow_corner_cutting=CORNER_CUTTING, name=name)

def map3_grid():
    occ = np.zeros((100, 100), dtype=bool)
    for x in MAP3_XS:
        for y in MAP3_YS:
            if (x, y) not in MAP3_MISSING:
                occ[y - 1:y + 4, x - 1:x + 4] = True
    return GridMap(occ, allow_corner_cutting=CORNER_CUTTING, name="paper_map3")

MAPS = {"30x30": grid_from_rows(MAP1_ROWS, "paper_map1"), "60x60": grid_from_rows(MAP2_ROWS, "paper_map2"),
        "100x100": map3_grid()}
for size, grid in MAPS.items():
    print(size, grid)

In [ ]:
# Palette and a helper that draws a map with explored cells and a path
INK, INK2, MUTED, SURFACE, GRID_LINE = "#0b0b0b", "#52514e", "#8a8984", "#fcfcfb", "#e4e3df"
OBSTACLE, EXPLORED, BLUE, ORANGE = "#3a3936", "#cde2fb", "#2a78d6", "#eb6834"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9.5, "axes.edgecolor": GRID_LINE,
                     "figure.facecolor": "white", "axes.facecolor": SURFACE})

def free_start(grid, cell):
    """Table 5 starts at (4, 24) on map1, which is an obstacle in the paper's own Fig. 5
    (the paper planned from it anyway). We free that single cell for that experiment only."""
    if grid.is_free(*cell):
        return grid
    occ = grid.occ.copy(); occ[cell[1], cell[0]] = False
    return GridMap(occ, allow_corner_cutting=grid.allow_corner_cutting, name=grid.name)

def plan(name, grid, start, goal):
    s, t = from_paper(*start), from_paper(*goal)
    return replace(PLANNERS[name], los=LOS).plan(free_start(grid, s), s, t)

def draw(ax, grid, start, goal, planner=None, title="", color=BLUE):
    res = plan(planner, grid, start, goal) if planner else None
    grid = free_start(grid, from_paper(*start))
    img = np.zeros(grid.occ.shape, dtype=int)
    if res:
        for x, y in res.closed:
            img[y, x] = 1
    img[grid.occ] = 2
    ax.imshow(img, cmap=ListedColormap([SURFACE, EXPLORED, OBSTACLE]), vmin=0, vmax=2, origin="lower",
              interpolation="nearest", extent=(0.5, grid.width + 0.5, 0.5, grid.height + 0.5))
    if res:
        xs, ys = [c[0] + 1 for c in res.path], [c[1] + 1 for c in res.path]
        ax.plot(xs, ys, color=color, lw=1.8)
        if PLANNERS[planner].prune:
            ax.plot(xs, ys, "o", ms=2.8, color=color)
        title += f"\nL={res.length:.2f} m, N={res.expanded}, C={res.corners}"
    ax.plot(*start, "s", ms=6, color=INK)
    ax.plot(*goal, "*", ms=10, color=INK, mec=SURFACE)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (size, goal) in zip(axes, (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99)))):
    draw(ax, MAPS[size], (2, 2), goal, title=f"Digitised paper map {size}")
plt.show()

In [ ]:
# Validation: our baseline planners on the digitised maps vs the paper's published numbers
CHECKS = [  # (table, map, start, goal, algorithm, paper length, paper nodes, paper corners)
    ("T2", "30x30", (2, 2), (28, 28), "Dijkstra", 40.30, 632, 11), ("T2", "30x30", (2, 2), (28, 28), "A*", 40.41, 130, 9),
    ("T3", "60x60", (2, 2), (58, 58), "Dijkstra", 92.10, 1740, 13), ("T3", "60x60", (2, 2), (58, 58), "A*", 92.30, 716, 12),
    ("T4", "100x100", (2, 2), (99, 99), "Dijkstra", 146.41, 7520, 31), ("T4", "100x100", (2, 2), (99, 99), "A*", 146.56, 2191, 34),
    ("T5", "30x30", (28, 28), (2, 2), "A*", 40.30, 202, 13), ("T5", "30x30", (4, 24), (29, 2), "A*", 38.21, 250, 14),
    ("T5", "60x60", (58, 58), (2, 2), "A*", 92.08, 898, 19), ("T5", "60x60", (4, 54), (59, 2), "A*", 118.49, 1159, 27),
]
rows = []
for table, size, s, g, alg, pl, pn, pc in CHECKS:
    r = plan(alg, MAPS[size], s, g)
    rows.append({"table": table, "map": size, "start->goal": f"{s}->{g}", "algorithm": alg,
                 "length paper": pl, "length ours": round(r.length, 2),
                 "nodes paper": pn, "nodes ours": r.expanded, "corners paper": pc, "corners ours": r.corners})
display(pd.DataFrame(rows))

### 4.1 Our digitiser
This is the code that produced the maps above from the paper's figure. With Internet access, the next cell downloads the figure from the project repository, re-digitises it, and checks that the result is identical to the embedded maps. Without Internet it is skipped.

In [ ]:
from PIL import Image
from scipy import ndimage

# Pixel geometry of the three panels in data/paper_figure5_maps.png, measured from
# the image (map extents for map1/map3; border-wall centre lines for map2).
MAP1_BOX = (47, 194, 104, 252)          # x0, x1, y0, y1 of the 30 x 30 grid
MAP2_BORDER_CENTRES = (307.5, 452.0, 106.0, 250.5)   # x of col 1 / col 60, y of row 60 / row 1
MAP3_BOX = (565, 713, 104, 252)         # extent of the 100 x 100 grid
MAP3_BLOCK = 5                          # block size in cells (all 99 detected blocks are 5 x 5)
# Block-group offsets (cells) for the 4 column groups and 4 row groups, found by --search-map3.
MAP3_GROUP_OFFSETS = {"dx": (-1, 0, 0, 0), "dy": (0, 0, 1, 0)}
MAP3_GROUPS = ((0, 3), (3, 6), (6, 9), (9, 10))   # block columns / rows that move together


def blackness(img: Image.Image) -> np.ndarray:
    """Per-pixel max over RGB: black obstacles are low, white/grey/red/blue are high."""
    return np.asarray(img.convert("RGB")).astype(float).max(axis=2)


def digitise_map1(bk: np.ndarray) -> np.ndarray:
    x0, x1, y0, y1 = MAP1_BOX
    n = 30
    w, h = (x1 - x0 + 1) / n, (y1 - y0 + 1) / n
    occ = np.zeros((n, n), dtype=bool)                     # row 0 = top of the figure
    for r in range(n):
        for c in range(n):
            cx, cy = x0 + (c + 0.5) * w, y0 + (r + 0.5) * h
            patch = bk[int(np.floor(cy - h / 4)):int(np.ceil(cy + h / 4)) + 1,
                       int(np.floor(cx - w / 4)):int(np.ceil(cx + w / 4)) + 1]
            occ[r, c] = patch.mean() < 128
    return occ


def digitise_map2(img: Image.Image) -> np.ndarray:
    n, s = 60, 10
    xl, xr, yt, yb = MAP2_BORDER_CENTRES
    pitch_x, pitch_y = (xr - xl) / (n - 1), (yb - yt) / (n - 1)
    left, top = int(xl - pitch_x), int(yt - pitch_y)
    right, bottom = int(xr + pitch_x) + 1, int(yb + pitch_y) + 1
    crop = img.convert("RGB").crop((left, top, right, bottom))
    big = blackness(crop.resize(((right - left) * s, (bottom - top) * s), Image.BILINEAR))
    occ = np.zeros((n, n), dtype=bool)
    half = 0.3                                              # sample +-0.3 cell around the centre
    for r in range(n):
        for c in range(n):
            cx = (xl + c * pitch_x - left) * s
            cy = (yt + r * pitch_y - top) * s
            patch = big[int(cy - half * pitch_y * s):int(cy + half * pitch_y * s) + 1,
                        int(cx - half * pitch_x * s):int(cx + half * pitch_x * s) + 1]
            occ[r, c] = patch.mean() < 140
    return occ


def detect_map3_blocks(bk: np.ndarray):
    """Lower-left corners (paper coordinates) of the 5 x 5 blocks detected in panel (c).

    Returns the sorted distinct block x and y origins and the set of (x, y) lattice
    positions where no block is drawn (the top row has no 10th column)."""
    x0, x1, y0, y1 = MAP3_BOX
    n = 100
    w, h = (x1 - x0 + 1) / n, (y1 - y0 + 1) / n
    labels, _ = ndimage.label(bk[y0:y1 + 1, x0:x1 + 1] < 110)
    origins = set()
    for ys, xs in ndimage.find_objects(labels):
        if ys.stop - ys.start < 3 or xs.stop - xs.start < 3:      # ignore specks
            continue
        col = int(round(xs.start / w)) + 1                         # 1-based paper x of the left column
        top_row = int(round(ys.start / h))                         # 0-based row from the top
        origins.add((col, n - top_row - MAP3_BLOCK + 1))           # paper y of the bottom row
    bx, by = sorted({o[0] for o in origins}), sorted({o[1] for o in origins})
    missing = {(x, y) for x in bx for y in by if (x, y) not in origins}
    return bx, by, missing


def build_map3(bx, by, missing, dx=(0, 0, 0, 0), dy=(0, 0, 0, 0)) -> np.ndarray:
    """Top-down occupancy array of the block lattice with per-group offsets applied."""
    occ = np.zeros((100, 100), dtype=bool)                         # [y-1, x-1], bottom-up
    for gi, (a, b) in enumerate(MAP3_GROUPS):
        for i in range(a, b):
            for gj, (c, d) in enumerate(MAP3_GROUPS):
                for j in range(c, d):
                    if (bx[i], by[j]) in missing:
                        continue
                    x, y = bx[i] + dx[gi], by[j] + dy[gj]
                    occ[y - 1:y - 1 + MAP3_BLOCK, x - 1:x - 1 + MAP3_BLOCK] = True
    return occ[::-1]


def _baseline_error(args):
    bx, by, missing, offsets = args
    grid = GridMap(build_map3(bx, by, missing, offsets[:4], offsets[4:])[::-1].copy(), allow_corner_cutting=True)
    s, t = from_paper(2, 2), from_paper(99, 99)
    if not (grid.is_free(*s) and grid.is_free(*t)):
        return None
    d, a = dijkstra(grid, s, t), astar(grid, s, t)
    err = (abs(np.log(d.expanded / 7520)) + abs(np.log(a.expanded / 2191))
           + 3 * abs(np.log(d.cost / 146.41)))                     # paper Table 4 baselines
    return err, sum(map(abs, offsets)), offsets


def search_map3_offsets(bx, by, missing):
    """Try every +-1 offset of the 8 block groups; prefer the best fit, then the fewest moves."""
    import itertools
    from multiprocessing import Pool
    jobs = [(bx, by, missing, o) for o in itertools.product((-1, 0, 1), repeat=8)]
    with Pool() as pool:
        results = [r for r in pool.map(_baseline_error, jobs, chunksize=64) if r]
    err, _, best = min(results, key=lambda r: (round(r[0], 6), r[1]))
    print(f"map3 offset search: best error {err:.4f} with dx={best[:4]}, dy={best[4:]}")
    return {"dx": tuple(best[:4]), "dy": tuple(best[4:])}


def digitise_map3(bk: np.ndarray, search: bool = False) -> np.ndarray:
    bx, by, missing = detect_map3_blocks(bk)
    offsets = search_map3_offsets(bx, by, missing) if search else MAP3_GROUP_OFFSETS
    return build_map3(bx, by, missing, offsets["dx"], offsets["dy"])

In [ ]:
FIG5_URL = ("https://raw.githubusercontent.com/Mariam1543/aqqal123/claude/relaxed-johnson-4f4051/"
            "adaptive-astar-reimplementation/data/paper_figure5_maps.png")
try:
    import urllib.request
    request = urllib.request.Request(FIG5_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=20) as resp:
        fig5 = Image.open(io.BytesIO(resp.read()))
except Exception as exc:
    fig5 = None
    print("Figure not downloaded (no Internet?) - digitiser check skipped:", type(exc).__name__)
if fig5 is not None:
    bk = blackness(fig5)
    redone = {"30x30": digitise_map1(bk)[::-1], "60x60": digitise_map2(fig5)[::-1], "100x100": digitise_map3(bk)[::-1]}
    for size, occ in redone.items():
        print(f"{size}: re-digitised map identical to the embedded map -> {np.array_equal(occ, MAPS[size].occ)}")
    plt.figure(figsize=(9, 3.5)); plt.imshow(fig5); plt.axis("off"); plt.title("Paper Fig. 5 (input of the digitiser)"); plt.show()

In [ ]:
# Generator parameters for the random maps of Part 9 (calibrated earlier; see archive/calibrate_maps.py)
CALIBRATION = {
    "map1": {
        "seed": 1796752687,
        "params": {
            "n_blobs": 17,
            "blob_cells": [
                4,
                35
            ]
        }
    },
    "map2": {
        "seed": 2048598178,
        "params": {
            "corridor": 4,
            "wall": 3,
            "loop_prob": 0.204
        }
    },
    "map3": {
        "seed": 694909934,
        "params": {
            "panel_h": 2,
            "aisle": 2,
            "panel_len": [
                5,
                7
            ],
            "gap": [
                2,
                4
            ],
            "missing_prob": 0.296,
            "clutter": 6
        }
    }
}

## 5. The paper's published results (Tables 1–5)

In [ ]:
PUBLISHED_CSV = """
table,map,start,goal,algorithm,time_s,length_m,expanded_nodes,corners_ge45
T1,30x30,2;2,28;28,A*,0.04,40.41,130,9
T1,30x30,2;2,28;28,Adaptive estimation function,0.03,40.20,118,13
T1,30x30,2;2,28;28,Distance model optimization,0.02,41.45,45,9
T1,30x30,2;2,28;28,Redundant point deletion,0.02,38.74,130,5
T1,60x60,2;2,58;58,A*,0.18,92.30,716,12
T1,60x60,2;2,58;58,Adaptive estimation function,0.09,92.05,412,12
T1,60x60,2;2,58;58,Distance model optimization,0.12,91.08,691,10
T1,60x60,2;2,58;58,Redundant point deletion,0.12,87.49,691,7
T2,30x30,2;2,28;28,A*,0.04,40.41,130,9
T2,30x30,2;2,28;28,Dijkstra [28],0.14,40.30,632,11
T2,30x30,2;2,28;28,Chen et al. [29],0.03,41.69,129,9
T2,30x30,2;2,28;28,Bidirectional A* [30],0.03,42.65,156,9
T2,30x30,2;2,28;28,Zhang et al. [31],0.04,40.29,98,11
T2,30x30,2;2,28;28,Adaptive A*,0.01,39.72,36,7
T3,60x60,2;2,58;58,A*,0.18,92.30,716,12
T3,60x60,2;2,58;58,Dijkstra [28],0.39,92.10,1740,13
T3,60x60,2;2,58;58,Chen et al. [29],0.12,92.08,696,12
T3,60x60,2;2,58;58,Bidirectional A* [30],0.18,93.91,704,12
T3,60x60,2;2,58;58,Zhang et al. [31],0.24,92.11,523,10
T3,60x60,2;2,58;58,Adaptive A*,0.06,90.89,268,8
T4,100x100,2;2,99;99,A*,0.70,146.56,2191,34
T4,100x100,2;2,99;99,Dijkstra [28],1.93,146.41,7520,31
T4,100x100,2;2,99;99,Chen et al. [29],0.52,146.56,2168,34
T4,100x100,2;2,99;99,Bidirectional A* [30],0.38,148.38,1126,32
T4,100x100,2;2,99;99,Zhang et al. [31],0.45,146.56,891,30
T4,100x100,2;2,99;99,Adaptive A*,0.25,140.67,113,14
T5,30x30,28;28,2;2,A*,0.03,40.30,202,13
T5,30x30,28;28,2;2,Adaptive A*,0.01,39.92,43,6
T5,30x30,4;24,29;2,A*,0.04,38.21,250,14
T5,30x30,4;24,29;2,Adaptive A*,0.01,37.68,33,5
T5,60x60,58;58,2;2,A*,0.25,92.08,898,19
T5,60x60,58;58,2;2,Adaptive A*,0.04,88.81,123,7
T5,60x60,4;54,59;2,A*,0.41,118.49,1159,27
T5,60x60,4;54,59;2,Adaptive A*,0.13,114.96,478,13
"""
PUBLISHED = pd.read_csv(io.StringIO(PUBLISHED_CSV.strip()))
PUBLISHED["algorithm"] = PUBLISHED["algorithm"].replace({"Dijkstra [28]": "Dijkstra",
                                                          "Bidirectional A* [30]": "Bidirectional A*"})
display(PUBLISHED)

### 5.1 Our audit of the paper's published numbers
Before comparing against the paper, we checked whether its own numbers are internally consistent.

In [ ]:
# (a) Abstract claims recomputed from Tables 2-4
t = PUBLISHED[PUBLISHED["table"].isin(["T2", "T3", "T4"])].set_index(["map", "algorithm"])
maps = ["30x30", "60x60", "100x100"]
time_red = np.mean([100 * (t.loc[(m, "A*"), "time_s"] - t.loc[(m, "Adaptive A*"), "time_s"]) / t.loc[(m, "A*"), "time_s"] for m in maps])
len_red = np.mean([t.loc[(m, "A*"), "length_m"] - t.loc[(m, "Adaptive A*"), "length_m"] for m in maps])
cor_red = np.mean([100 * (t.loc[(m, "A*"), "corners_ge45"] - t.loc[(m, "Adaptive A*"), "corners_ge45"]) / t.loc[(m, "A*"), "corners_ge45"] for m in maps])
print(f"Abstract: 69% time, 2.66 m, 38.1% corners  ->  recomputed: {time_red:.2f}%, {len_red:.2f} m, {cor_red:.1f}%  (consistent)")

# (b) Claims in the text vs the paper's own tables
def pct(a, b):
    return 100 * (a - b) / a
claims = pd.DataFrame([
    ("T1 30x30: adaptive weight 'nodes dropped to 86.2%'", "86.2%", f"{pct(130, 118):.1f}% drop (130 -> 118)"),
    ("T2: Dijkstra 'increase of 79.4%' nodes vs A*", "+79.4%", f"+{100 * (632 - 130) / 130:.0f}% (79.4% is the reduction relative to Dijkstra)"),
    ("T5 60x60 swapped: time reduction '88.9%'", "88.9%", f"{pct(0.25, 0.04):.1f}% (0.25 -> 0.04 s)"),
    ("T5 30x30 swapped: node reduction '86.8%'", "86.8%", f"{pct(202, 43):.1f}% (202 -> 43); 86.8% is the random-pair row"),
], columns=["Statement in the paper", "Stated", "Computed from the paper's own table"])
display(claims)

# (c) Optimality check: an admissible A* must give the same length as Dijkstra, in both directions
rows = []
for m in maps:
    rows.append({"map": m, "paper A* length": t.loc[(m, "A*"), "length_m"],
                 "paper Dijkstra length": t.loc[(m, "Dijkstra"), "length_m"],
                 "A* longer than Dijkstra?": t.loc[(m, "A*"), "length_m"] > t.loc[(m, "Dijkstra"), "length_m"]})
display(pd.DataFrame(rows).set_index("map"))
t5 = PUBLISHED[(PUBLISHED["table"] == "T5") & (PUBLISHED["algorithm"] == "A*")]
print("A* length when start and goal are swapped (should be identical to the forward run):")
print(f"  30x30: forward 40.41 vs swapped {t5.iloc[0]['length_m']:.2f};  60x60: forward 92.30 vs swapped {t5.iloc[2]['length_m']:.2f}")

# (d) Can the published lengths be built from 8-connected grid steps (1 and sqrt 2)?
def is_grid_length(L, dx, dy, tol=0.01):
    for b in range(0, 400):
        a = L - b * math.sqrt(2)
        if a < -tol:
            return False
        A = round(a)
        if abs(a - A) <= tol and A + 2 * b >= dx + dy and A + b >= max(dx, dy) and (A - dx - dy) % 2 == 0:
            return True
    return False
checks = [(m, alg, t.loc[(m, alg), "length_m"], d) for m, d in (("30x30", 26), ("60x60", 56), ("100x100", 97))
          for alg in ("A*", "Dijkstra")]
display(pd.DataFrame([{"map": m, "algorithm": a, "published length": L, "valid 8-connected grid length?": is_grid_length(L, d, d)}
                      for m, a, L, d in checks]))
print("Conclusion: the paper's baseline A* is not optimal under its own length metric, and its length metric is "
      "not plain grid-step cost -> exact lengths cannot be matched; we compare changes relative to A*.")

# (e) Table 5 starts at (4, 24) on map1 - is that cell free in the paper's own Fig. 5?
print("\nTable 5 start (4, 24) is free in the digitised Fig. 5:", MAPS["30x30"].is_free(*from_paper(4, 24)),
      "-> the paper planned from an obstacle cell (we free that one cell for that experiment).")

## 6. Experiments

Each experiment runs on the digitised paper maps with the paper's start/goal points, corner cutting allowed and Bresenham line of sight (D2, D6).
- **Length, nodes and corners** are deterministic, identical on every run and every machine.
- **Time** is the median of repeated runs. The paper used MATLAB and we use Python, so absolute times are not comparable. We compare the **change relative to A\*** instead.

In [ ]:
METRICS = ["time_s", "length_m", "expanded_nodes", "corners_ge45"]
PUBLISHED_ONLY = {"Chen et al. [29]", "Zhang et al. [31]"}   # not described in the paper -> published numbers only

def run_case(name, size, start, goal):
    spec = replace(PLANNERS[name], los=LOS)
    s, t = from_paper(*start), from_paper(*goal)
    grid = free_start(MAPS[size], s)
    res = spec.plan(grid, s, t)
    timing = time_planner(spec, grid, s, t, repeats=TIMING_REPEATS)
    return {"algorithm": name, "time_s": timing["time_median"], "length_m": res.length,
            "expanded_nodes": res.expanded, "corners_ge45": res.corners, "raw_cost": res.raw_cost}

def relative(df):
    """Change relative to the A* row: metres for length, percent for the rest."""
    base = df[df["algorithm"] == "A*"].iloc[0]
    out = df[["algorithm"]].copy()
    for m in METRICS:
        out[m] = df[m] - base[m] if m == "length_m" else 100 * (df[m] - base[m]) / base[m]
    return out

def experiment(table, size, start, goal, algorithms):
    """Run one experiment and show paper vs ours (absolute values and change vs A*)."""
    ours = pd.DataFrame([run_case(a, size, start, goal) for a in algorithms])
    key = f"{start[0]};{start[1]}"
    pub = PUBLISHED[(PUBLISHED["table"] == table) & (PUBLISHED["map"] == size)
                    & (PUBLISHED["start"] == key) & (PUBLISHED["goal"] == f"{goal[0]};{goal[1]}")]
    merged = pub[["algorithm"] + METRICS].merge(ours[["algorithm"] + METRICS], on="algorithm",
                                                 how="outer", suffixes=(" paper", " ours"))
    order = list(dict.fromkeys(list(pub["algorithm"]) + list(ours["algorithm"])))
    merged = merged.set_index("algorithm").loc[order]
    absolute = pd.DataFrame(index=merged.index)
    for m in METRICS:
        absolute[f"{m} paper"] = merged[f"{m} paper"]
        absolute[f"{m} ours"] = merged[f"{m} ours"]
        if m != "time_s":
            absolute[f"{m} diff"] = merged[f"{m} ours"] - merged[f"{m} paper"]
    rel_p = relative(pub.reset_index(drop=True)).set_index("algorithm").add_suffix(" paper")
    rel_o = relative(ours).set_index("algorithm").add_suffix(" ours")
    rel = rel_p.join(rel_o, how="outer").reindex(order).drop(index="A*")
    rel = rel[[f"{m} {w}" for m in METRICS for w in ("paper", "ours")]]
    display(Markdown(f"**{table} — {size}, start {start} → goal {goal}** (absolute values)"))
    display(absolute.style.format(precision=4, na_rep="—"))
    display(Markdown("Change relative to A\\* (length in m, others in %; negative = better):"))
    display(rel.style.format("{:+.1f}", na_rep="—"))
    return ours.assign(table=table, map=size, start=str(start), goal=str(goal))

ALL_RESULTS = []

### 6.1 Table 1 — Ablation of the three improvements
- **AEF** = adaptive estimation function only.
- **DMO** = hybrid distance model only.
- **RPD** = redundant point deletion only.

We also add the full combination and a single-pass pruning variant.

In [ ]:
ABLATION = ["A*", "Adaptive estimation function", "Distance model optimization", "Redundant point deletion",
            "AEF+DMO (no pruning)", "Adaptive A*"]
ALL_RESULTS.append(experiment("T1", "30x30", (2, 2), (28, 28), ABLATION))

In [ ]:
ALL_RESULTS.append(experiment("T1", "60x60", (2, 2), (58, 58), ABLATION))

**Observation — two rows of the paper's Table 1 (60×60) appear to be swapped, and its pruning row is not isolated.**

On 30×30 our isolated components match the paper almost exactly: adaptive weight 119 vs 118 nodes, and hybrid distance 46 vs 45 nodes with 41.46 vs 41.45 m.

On 60×60, however, three pieces of evidence point to a problem in the paper's table:
1. **Node counts.** Our *adaptive-weight* run matches the paper's *distance-model* row, and vice versa (table below). The same code matches both rows on 30×30.
2. **Theory.** ω is below 1 for nodes with d < 0.54·D and above 1 elsewhere, so the adaptive weight alone changes the search only a little. The paper's own 30×30 row shows −9%, so −42.5% on 60×60 is implausible.
3. **The pruning row copies another row.** Pruning is post-processing and cannot change the node count, so an isolated pruning row must show A\*'s count. On 30×30 it does (130 = A\*). On 60×60 it repeats the 691-node row exactly, in nodes and time (691, 0.12 s), instead of A\*'s 716. This suggests the pruning was applied on top of another change on 60×60 (cumulative) but alone on 30×30 (isolated).

In [ ]:
t1 = pd.concat(ALL_RESULTS)
t1 = t1[(t1["table"] == "T1") & (t1["map"] == "60x60")].set_index("algorithm")
p1 = PUBLISHED[(PUBLISHED["table"] == "T1") & (PUBLISHED["map"] == "60x60")].set_index("algorithm")
display(pd.DataFrame({
    "paper nodes": [p1.loc["Adaptive estimation function", "expanded_nodes"], p1.loc["Distance model optimization", "expanded_nodes"]],
    "our nodes, same row": [t1.loc["Adaptive estimation function", "expanded_nodes"], t1.loc["Distance model optimization", "expanded_nodes"]],
    "our nodes, other row": [t1.loc["Distance model optimization", "expanded_nodes"], t1.loc["Adaptive estimation function", "expanded_nodes"]],
}, index=["paper row: Adaptive estimation function", "paper row: Distance model optimization"]))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (name, title) in zip(axes, [("A*", "(a) A*"), ("Adaptive estimation function", "(b) Adaptive estimation"),
                                    ("Distance model optimization", "(c) Distance model"),
                                    ("Redundant point deletion", "(d) Redundant point deletion")]):
    draw(ax, MAPS["60x60"], (2, 2), (58, 58), name, title)
fig.suptitle("Ablation on the 60x60 map (cf. paper Fig. 7) - explored cells shaded", color=INK2)
plt.show()

### 6.2 Tables 2–4 — Comparison with other algorithms
Chen et al. [29] and Zhang et al. [31] are not described in the paper, so they appear with their **published numbers only**.

**A\* + pruning** and **A\* + octile + pruning** are extra baselines we added for a fair comparison.

In [ ]:
COMPARISON = ["A*", "Dijkstra", "Bidirectional A*", "Adaptive A*", "Redundant point deletion", "A*+Octile+RPD",
              "Adaptive A* (V4: E+M without 1/2)"]
ALL_RESULTS.append(experiment("T2", "30x30", (2, 2), (28, 28), COMPARISON))

In [ ]:
ALL_RESULTS.append(experiment("T3", "60x60", (2, 2), (58, 58), COMPARISON))

In [ ]:
ALL_RESULTS.append(experiment("T4", "100x100", (2, 2), (99, 99), COMPARISON))

In [ ]:
for size, goal in (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99))):
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
    for ax, (name, title) in zip(axes, [("A*", "(a) A*"), ("Dijkstra", "(b) Dijkstra"),
                                        ("Bidirectional A*", "(c) Bidirectional A*"),
                                        ("Redundant point deletion", "(d) A* + pruning"),
                                        ("Adaptive A*", "(e) Adaptive A*")]):
        draw(ax, MAPS[size], (2, 2), goal, name, title, ORANGE if name == "Adaptive A*" else BLUE)
    fig.suptitle(f"Algorithms on the {size} map (cf. paper Figs. 8-10)", color=INK2)
    plt.show()

### 6.3 Table 5 — Different start and end points
The 30×30 start (4,24) is an **obstacle in the paper's own Fig. 5**: the cell is solid black in the figure. The paper still reports a path from it. Our code **frees that single cell for this experiment only** (`free_start`); every planner uses the same modified map. The next cell shows why we free it instead of moving the start or skipping the case.

In [ ]:
g30, goal = MAPS["30x30"], from_paper(29, 2)
print("Is (4,24) free in the digitised Fig. 5?", g30.is_free(*from_paper(4, 24)))
print("Paper, A* from (4,24): 38.21 m / 250 nodes / 14 corners")
for label, start in (("free the cell (ours)", (4, 24)), ("move start to (3,24)", (3, 24)),
                     ("move start to (5,24)", (5, 24)), ("move start to (4,23)", (4, 23))):
    res = PLANNERS["A*"].plan(free_start(g30, from_paper(*start)), from_paper(*start), goal)
    print(f"  {label:22s}: {res.length:.2f} m / {res.expanded} nodes / {res.corners} corners")
print("Only freeing the cell reproduces the paper exactly, which suggests the authors' planner treated it as free.")

In [ ]:
for size, start, goal in (("30x30", (28, 28), (2, 2)), ("30x30", (4, 24), (29, 2)),
                          ("60x60", (58, 58), (2, 2)), ("60x60", (4, 54), (59, 2))):
    ALL_RESULTS.append(experiment("T5", size, start, goal, ["A*", "Adaptive A*", "Adaptive A* (V4: E+M without 1/2)"]))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for col, (size, s, g) in enumerate((("30x30", (28, 28), (2, 2)), ("30x30", (4, 24), (29, 2)),
                                    ("60x60", (58, 58), (2, 2)), ("60x60", (4, 54), (59, 2)))):
    draw(axes[0, col], MAPS[size], s, g, "A*", f"A* {s}->{g}")
    draw(axes[1, col], MAPS[size], s, g, "Adaptive A*", f"Adaptive A* {s}->{g}", ORANGE)
fig.suptitle("Different start and end points (cf. paper Figs. 11-12)", color=INK2)
plt.show()

## 7. Paired timing study (planning-time metric)
A single run takes well under a millisecond, so one-off timings are noisy. Here A\* and Adaptive A\* are run **alternately**, many times, and we report the median of the paired ratios with a bootstrap 95% confidence interval.

In [ ]:
def paired_timing(size, goal, pairs=TIMING_PAIRS, seed=0):
    grid, s, t = MAPS[size], from_paper(2, 2), from_paper(*goal)
    a_spec, b_spec = PLANNERS["A*"], replace(PLANNERS["Adaptive A*"], los=LOS)
    for _ in range(5):
        a_spec.plan(grid, s, t); b_spec.plan(grid, s, t)
    ta, tb = [], []
    gc.disable()
    try:
        for i in range(pairs):
            for spec in ((a_spec, b_spec) if i % 2 == 0 else (b_spec, a_spec)):
                t0 = time.perf_counter(); spec.plan(grid, s, t)
                (ta if spec is a_spec else tb).append(time.perf_counter() - t0)
    finally:
        gc.enable()
    ratio = np.array(tb) / np.array(ta)
    rng = np.random.default_rng(seed)
    lo, hi = np.percentile([np.median(rng.choice(ratio, ratio.size)) for _ in range(2000)], [2.5, 97.5])
    return {"map": size, "A* median (ms)": 1e3 * np.median(ta), "Adaptive A* median (ms)": 1e3 * np.median(tb),
            "ratio": np.median(ratio), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "time reduction ours (%)": 100 * (1 - np.median(ratio))}

TIMING = pd.DataFrame([paired_timing(s, g) for s, g in (("30x30", (28, 28)), ("60x60", (58, 58)),
                                                        ("100x100", (99, 99)))])
TIMING["time reduction paper (%)"] = [75.0, 66.7, 64.3]
display(TIMING.set_index("map").round(3))

## 8. Headline comparison — published vs. reproduced (assignment table)

In [ ]:
RESULTS = pd.concat(ALL_RESULTS, ignore_index=True)

def mean_reduction(df_rows, metric):
    vals = []
    for size in ("30x30", "60x60", "100x100"):
        r = df_rows[df_rows["map"] == size].set_index("algorithm")
        a, b = r.loc["A*", metric], r.loc["Adaptive A*", metric]
        vals.append(a - b if metric == "length_m" else 100 * (a - b) / a)
    return np.mean(vals), vals

ours_tab = RESULTS[RESULTS["table"].isin(["T2", "T3", "T4"])]
paper_tab = PUBLISHED[PUBLISHED["table"].isin(["T2", "T3", "T4"])]
rows = []
for label, metric, unit in (("Metric 1: planning-time reduction vs A*", "time_s", "%"),
                            ("Metric 2: path-length reduction vs A*", "length_m", "m"),
                            ("Metric 3: expanded-node reduction vs A*", "expanded_nodes", "%"),
                            ("Metric 4: reduction of corners >= 45 deg vs A*", "corners_ge45", "%")):
    p, p_each = mean_reduction(paper_tab, metric)
    if metric == "time_s":   # paired timing study is the reliable time measurement
        o_each = list(TIMING["time reduction ours (%)"]); o = float(np.mean(o_each))
    else:
        o, o_each = mean_reduction(ours_tab, metric)
    fmt = (lambda v: f"{v:.2f} m") if unit == "m" else (lambda v: f"{v:.1f}%")
    rows.append({"Evaluation metric": label,
                 "Published result": f"{fmt(p)}  ({', '.join(fmt(v) for v in p_each)})",
                 "Our result": f"{fmt(o)}  ({', '.join(fmt(v) for v in o_each)})",
                 "Difference": f"{o - p:+.2f} m" if unit == "m" else f"{o - p:+.1f} pp"})
HEADLINE = pd.DataFrame(rows).set_index("Evaluation metric")
display(HEADLINE)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
sizes = ["30x30", "60x60", "100x100"]
for ax, (metric, title) in zip(axes, (("time_s", "Planning time (% vs A*)"),
                                      ("expanded_nodes", "Expanded nodes (% vs A*)"),
                                      ("length_m", "Path length (m vs A*)"),
                                      ("corners_ge45", "Corners >= 45 deg (% vs A*)"))):
    p = [-v for v in mean_reduction(paper_tab, metric)[1]]
    o = ([-v for v in TIMING["time reduction ours (%)"]] if metric == "time_s"
         else [-v for v in mean_reduction(ours_tab, metric)[1]])
    x = np.arange(3)
    ax.bar(x - 0.19, p, 0.36, color=BLUE, label="Published (paper)")
    ax.bar(x + 0.19, o, 0.36, color=ORANGE, label="Reproduced (ours)")
    ax.axhline(0, color=MUTED, lw=0.8); ax.set_xticks(x, sizes); ax.set_title(title)
    ax.grid(axis="y", color=GRID_LINE); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (BLUE, ORANGE)]
fig.legend(handles, ["Published (paper)", "Reproduced (ours)"], frameon=False, ncol=2,
           loc="upper right", bbox_to_anchor=(0.99, 1.08))
fig.suptitle("Adaptive A* relative to A*: published vs reproduced (negative = better)",
             color=INK2, x=0.01, ha="left", y=1.06)
plt.show()

## 9. Robustness on random maps (multiple runs)
The algorithms are deterministic, so the meaningful "multiple runs" vary the **map**. We use the same calibrated generators with **unseen seeds**: 50 maps per scene type, 150 in total. Each method is compared with A\* on the same map using the paired **Wilcoxon signed-rank test**.

In [ ]:
ROB_PAIRS = {"map1": (2, 2, 28, 28), "map2": (2, 2, 58, 58), "map3": (2, 2, 99, 99)}
rob_rows = []
for map_id, (sx, sy, gx, gy) in ROB_PAIRS.items():
    params = {k: tuple(v) if isinstance(v, list) else v for k, v in CALIBRATION[map_id]["params"].items()}
    seed, made = 0, 0
    while made < ROBUST_MAPS:
        seed += 1
        grid = make_map(map_id, 900_000 + seed, **params)
        if grid is None:
            continue
        grid.allow_corner_cutting = CORNER_CUTTING        # same settings as the paper maps
        made += 1
        s, t = from_paper(sx, sy), from_paper(gx, gy)
        for name in ("A*", "Redundant point deletion", "Adaptive A*"):
            spec = replace(PLANNERS[name], los=LOS)
            res = spec.plan(grid, s, t)
            tm = time_planner(spec, grid, s, t, repeats=5)
            rob_rows.append({"map": map_id, "seed": 900_000 + seed, "algorithm": name, "time_s": tm["time_median"],
                             "length_m": res.length, "expanded_nodes": res.expanded,
                             "corners_ge45": res.corners, "raw_cost": res.raw_cost})
ROB = pd.DataFrame(rob_rows)
print(f"{ROB.groupby(['map', 'seed']).ngroups} random maps evaluated ({ROBUST_MAPS} per scene type)")

In [ ]:
summary = []
for map_id in ROB_PAIRS:
    base = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == "A*")].set_index("seed")
    for name in ("Redundant point deletion", "Adaptive A*"):
        other = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == name)].set_index("seed")
        for m in METRICS:
            a, b = base[m].to_numpy(float), other.loc[base.index, m].to_numpy(float)
            change = b - a if m == "length_m" else 100 * (b - a) / a
            p = wilcoxon(a, b).pvalue if np.any(a != b) else 1.0
            summary.append({"map": map_id, "method": "A* + pruning" if name != "Adaptive A*" else name,
                            "metric": m, "mean change": change.mean(), "std": change.std(ddof=1),
                            "better/equal/worse": f"{(b < a).sum()}/{(b == a).sum()}/{(b > a).sum()}",
                            "Wilcoxon p": p})
display(pd.DataFrame(summary).style.format({"mean change": "{:+.2f}", "std": "{:.2f}", "Wilcoxon p": "{:.2g}"}))

# Where do the length gains come from?
a = ROB[ROB["algorithm"] == "A*"].set_index(["map", "seed"])
b = ROB[ROB["algorithm"] == "Adaptive A*"].set_index(["map", "seed"])
c = ROB[ROB["algorithm"] == "Redundant point deletion"].set_index(["map", "seed"])
gap = 100 * (b["raw_cost"] - a["raw_cost"]) / a["raw_cost"]
diff = b["length_m"] - c["length_m"]
display(pd.DataFrame({
    "raw Adaptive A* path suboptimal (maps)": (gap > 1e-9).groupby(level=0).sum(),
    "mean / max suboptimality (%)": gap.groupby(level=0).agg(lambda g: f"{g.mean():.2f} / {g.max():.2f}"),
    "Adaptive A* longer than A*+pruning (maps)": (diff > 1e-9).groupby(level=0).sum(),
    "mean length difference (m)": diff.groupby(level=0).mean().round(2),
}))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, m in zip(axes, METRICS):
    for k, (map_id, size) in enumerate((("map1", "30x30"), ("map2", "60x60"), ("map3", "100x100"))):
        base = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == "A*")].set_index("seed")[m]
        for j, (name, color) in enumerate((("Redundant point deletion", BLUE), ("Adaptive A*", ORANGE))):
            v = ROB[(ROB["map"] == map_id) & (ROB["algorithm"] == name)].set_index("seed")[m]
            change = (v - base) if m == "length_m" else 100 * (v - base) / base
            bp = ax.boxplot(change.to_numpy(), positions=[k * 1.2 + j * 0.42], widths=0.34, patch_artist=True,
                            medianprops={"color": INK}, flierprops={"marker": "o", "ms": 2.5,
                                                                    "mfc": color, "mec": color})
            bp["boxes"][0].set(facecolor=color, edgecolor=color, alpha=0.85)
    ax.set_xticks([k * 1.2 + 0.21 for k in range(3)], ["30x30", "60x60", "100x100"])
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title({"time_s": "Time (% vs A*)", "length_m": "Length (m vs A*)",
                  "expanded_nodes": "Expanded nodes (% vs A*)", "corners_ge45": "Corners (% vs A*)"}[m])
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (BLUE, ORANGE)]
fig.legend(handles, ["A* + pruning", "Adaptive A*"], frameon=False, ncol=2,
           loc="upper right", bbox_to_anchor=(0.99, 1.08))
fig.suptitle("Change relative to A* on random maps (negative = better)", color=INK2, x=0.01, ha="left", y=1.06)
plt.show()

## 10. A hypothesis: what did the authors' code run? (sensitivity analysis)

This part is **not our main result**. Our main results (Parts 6–9) follow Eqs. 5–7 **exactly as printed**. On the digitised maps, the baselines and each improvement *alone* reproduce the paper almost exactly (Parts 4 and 6.1). The **full** Adaptive A\* is the exception: it expands clearly more nodes than published.

The table tests the two remaining ambiguities on every published Adaptive A\* case:
- **(i) Pruning order:** anchor-based vs single-pass three-node window.
- **(ii) Heuristic:** Eq. 7 as printed, (E+M)/2, vs **V4**, E+M without the ½.

In [ ]:
VARIANTS = {
    "as printed": PlannerSpec("x", heuristic="hybrid", adaptive=True, prune=True, los=LOS),
    "as printed, single-pass": PlannerSpec("x", heuristic="hybrid", adaptive=True, prune=True, prune_mode="single-pass", los=LOS),
    "V4 (E+M)": PlannerSpec("x", heuristic="hybrid_sum", adaptive=True, prune=True, los=LOS),
    "V4, single-pass": PlannerSpec("x", heuristic="hybrid_sum", adaptive=True, prune=True, prune_mode="single-pass", los=LOS),
}
rows = []
for _, r in PUBLISHED[PUBLISHED["algorithm"] == "Adaptive A*"].iterrows():
    s = from_paper(*map(int, r["start"].split(";"))); t = from_paper(*map(int, r["goal"].split(";")))
    grid = free_start(MAPS[r["map"]], s)
    row = {"case": f"{r['table']} {r['map']} {r['start']}->{r['goal']}",
           "paper N / L / C": f"{r['expanded_nodes']} / {r['length_m']:.2f} / {r['corners_ge45']}"}
    for name, spec in VARIANTS.items():
        res = spec.plan(grid, s, t)
        row[name] = f"{res.expanded} / {res.length:.2f} / {res.corners}"
    rows.append(row)
display(Markdown("Each cell: expanded nodes / path length (m) / corners >= 45 deg"))
display(pd.DataFrame(rows).set_index("case"))

In [ ]:
# Headline metrics again, now for V4 (cf. Part 8). Time here is the 30-run median.
v4 = RESULTS[RESULTS["table"].isin(["T2", "T3", "T4"])]
rows = []
for label, metric, unit in (("planning-time reduction", "time_s", "%"), ("path-length reduction", "length_m", "m"),
                            ("expanded-node reduction", "expanded_nodes", "%"), ("corners >= 45 deg reduction", "corners_ge45", "%")):
    vals = []
    for size in ("30x30", "60x60", "100x100"):
        r = v4[v4["map"] == size].set_index("algorithm")
        a, b = r.loc["A*", metric], r.loc["Adaptive A* (V4: E+M without 1/2)", metric]
        vals.append(a - b if unit == "m" else 100 * (a - b) / a)
    p = mean_reduction(paper_tab, metric)[0]
    rows.append({"metric": label, "published": f"{p:.2f} m" if unit == "m" else f"{p:.1f}%",
                 "V4 (ours)": f"{np.mean(vals):.2f} m" if unit == "m" else f"{np.mean(vals):.1f}%"})
display(pd.DataFrame(rows).set_index("metric"))

**Reading the table:**
- **Expanded nodes.** With V4 they match the published Adaptive A\* closely on every case (e.g. 40 vs 36, 261 vs 268, 114 vs 113, 35 vs 33). With the equations as printed they do not (66, 370, 448, 93). The published numbers are therefore *consistent with* the authors' full method using E+M without the ½, even though Eq. 7 and their isolated distance-model row both use the ½.
- **Why this is a hypothesis, not proof.** ω·(E+M) equals 2·ω·hybrid, so a missing ½ cannot be told apart from an extra weight of 2. Undocumented details such as tie-breaking could also contribute, and we do not have the authors' code.
- **Why V4 expands so few nodes.** It overestimates the true cost by up to 3.17× (Part 2.2), so it behaves almost like greedy best-first search.
- **Path lengths.** They are close to the paper with both readings.
- **Corners.** The paper's counts sit between the anchor and single-pass pruning readings, and no single reading reproduces all of them. The pruning step is under-specified.

### 10.1 Second line of evidence: the paper's own result figures

The paper's Figs. 8 and 9 shade every expanded cell grey. Classifying each cell of those panels (repository script `scripts/compare_paper_figures.py`) gives:

| Paper panel | Our planner | Our nodes | Paper-explored cells we also expand | Wrongly expanded: cells we expand that are **white (never explored)** in the paper's figure |
|---|---|---|---|---|
| Fig. 8(a) 30×30 | A\* | 130 | 99/99 (100%) | **0** |
| Fig. 8(b) 30×30 | Dijkstra | 633 | 600/600 (100%) | **0** |
| Fig. 8(f) 30×30 | Adaptive A\*, Eqs. as printed (main) | 66 | 12/18 | 29 |
| Fig. 8(f) 30×30 | Adaptive A\*, V4 (hypothesis) | 40 | 13/18 | 2 |
| Fig. 9(a) 60×60 | A\* | 715 | (grid lines blur the shading) | **0** |
| Fig. 9(f) 60×60 | Adaptive A\*, Eqs. as printed (main) | 370 | (grid lines blur the shading) | 122 |
| Fig. 9(f) 60×60 | Adaptive A\*, V4 (hypothesis) | 261 | (grid lines blur the shading) | 2 |

- A\* and Dijkstra reproduce the paper's shaded regions cell for cell.
- With the equations as printed, our Adaptive A\* searches 29 (30×30) and 122 (60×60) cells that the authors' planner never explored. V4 searches only 2 such cells on each map. This is consistent with the V4 hypothesis, but it is not proof.

The figure below (downloaded from the repository if Internet is on) puts the paper's panels next to ours.

In [ ]:
PANELS_URL = ("https://raw.githubusercontent.com/Mariam1543/aqqal123/claude/relaxed-johnson-4f4051/"
              "adaptive-astar-reimplementation/figures/paper_maps/fig_paper_vs_ours_panels.png")
try:
    import urllib.request
    request = urllib.request.Request(PANELS_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=20) as resp:
        panels = Image.open(io.BytesIO(resp.read()))
    plt.figure(figsize=(12, 8.5)); plt.imshow(panels); plt.axis("off"); plt.show()
except Exception as exc:
    print("Comparison figure not downloaded (no Internet?):", type(exc).__name__)

### 10.2 Calibration disclosure and line-of-sight sensitivity

Some settings were chosen **because they made our numbers match the paper's**, which is calibration:

| Choice | Matched to | Uses the proposed method's numbers? |
|---|---|---|
| Corner cutting allowed | A\* and Dijkstra node counts | No |
| Euclidean baseline heuristic | A\* node counts | No |
| Map 3 block positions (±1 cell, 2 of 8 groups) | Table 4 Dijkstra nodes and length, A\* nodes | No |
| Bresenham line of sight | pruned lengths (Table 1 pruning rows, full method on 100×100) | **Yes**, so the strict supercover alternative is shown below |
| Table 5 start (4,24) freed | Table 5 A\* row | No |

Consequences:
- The **Adaptive A\* node counts were never used for calibration**. Expanded nodes do not depend on the line-of-sight test.
- The paper-map validation of Part 4 (e.g. "A\* expands 130 nodes") and the matching repository unit tests are therefore **map-regression checks**: they check the maps and settings, not the algorithm. The algorithm itself is checked on synthetic grids in Part 3.

In [ ]:
# Metrics 2 and 4 under both line-of-sight tests (Adaptive A* as printed vs A*, Tables 2-4)
rows = []
for los in ("bresenham", "supercover"):
    dl, dc = [], []
    for size, goal in (("30x30", (28, 28)), ("60x60", (58, 58)), ("100x100", (99, 99))):
        s, t = from_paper(2, 2), from_paper(*goal)
        a = PLANNERS["A*"].plan(MAPS[size], s, t)
        b = replace(PLANNERS["Adaptive A*"], los=los).plan(MAPS[size], s, t)
        dl.append(a.length - b.length); dc.append(100 * (a.corners - b.corners) / a.corners)
    rows.append({"line of sight": los + (" (main)" if los == LOS else ""),
                 "length reduction (m) 30/60/100": " / ".join(f"{v:.2f}" for v in dl), "mean (m)": round(np.mean(dl), 2),
                 "corner reduction (%) 30/60/100": " / ".join(f"{v:.1f}" for v in dc), "mean (%)": round(np.mean(dc), 1)})
rows.append({"line of sight": "paper", "length reduction (m) 30/60/100": "0.69 / 1.41 / 5.89", "mean (m)": 2.66,
             "corner reduction (%) 30/60/100": "22.2 / 33.3 / 58.8", "mean (%)": 38.1})
display(pd.DataFrame(rows).set_index("line of sight"))

## 11. Critical analysis (assignment Section 6)

A higher or lower number alone does not settle a verdict, so we judge each claim separately. Main results = Eqs. 5–7 **as printed**.

| Claim | Verdict | Why |
|---|---|---|
| A\* and Dijkstra baselines | **Comparable** | Within 1–5 nodes; the Table 5 pair is identical in every metric. Part of this agreement is by calibration (Part 10.2) |
| Each improvement alone (Table 1) | **Comparable** | 119 vs 118 and 46 vs 45 nodes, 41.46 vs 41.45 m; on 60×60 after the apparent row swap (Part 6.1) |
| Full Adaptive A\* as printed | **Partly comparable** | Faster, fewer nodes, shorter and smoother paths than A\* on every map, as claimed. But the size differs: nodes −59% vs −77%, time −40% vs −69%. The node gap is consistent with the V4 hypothesis (Part 10) |
| "The improved heuristic gives shorter / near-optimal paths" | **Not supported** | The full estimate overestimates by up to 58.5%, so the raw path is suboptimal on 90 of 150 random maps. A\* + pruning is as short or shorter on 110 of 150 (Part 9). The length and smoothness gains come from **pruning**; the heuristic buys **speed** |

**Per assignment metric** (mean over Tables 2–4, Adaptive A\* as printed vs A\*):

| Metric | Published | Ours (as printed) | Category |
|---|---|---|---|
| 1 Planning time | −68.7% | ≈ −40% (paired) | **(C) lower**: bounded by the node reduction, plus Python overhead; on 100×100 ours is higher (≈ 72% vs 64%) |
| 2 Path length | −2.66 m | −3.70 m (Bresenham), −1.29 m (supercover) | **(B)** or **(C)** depending on the line-of-sight test (Part 10.2); same sign |
| 3 Expanded nodes | −76.6% | ≈ −59% | **(C) lower**; ≈ −76% under the V4 hypothesis |
| 4 Corners ≥ 45° | −38.1% | −67.4% (Bresenham), −40.0% (supercover) | **(B)** or **(A)** depending on the line-of-sight test |

**Overall: partly reproduced.**
- The baselines and each improvement are comparable.
- The full method's advantages hold in direction but not in size when Eq. 7 is followed as printed.
- The claim that the heuristic itself shortens paths is not supported.

**Technical justification of the remaining differences:**
- **Dataset.** We use our digitisation of Fig. 5, with Map 3 calibrated on baseline statistics. One start cell is an obstacle in the paper's own figure; we free that cell for that experiment (Part 6.3).
- **Settings and parameters.** The paper never states that corner cutting is allowed or which line-of-sight test it uses. Both were inferred by matching published numbers, which we disclose as calibration (Part 10.2). The pruning order is still ambiguous: our corner counts differ from the paper with either reading (Part 10).
- **Randomness.** Length, nodes and corners are deterministic and exactly repeatable. Time is the only noisy metric, so we report medians and a paired study with confidence intervals. Robustness is tested on 150 random maps with Wilcoxon tests.
- **Software and hardware.** MATLAB on an i5 laptop vs Python on a cloud CPU. On small maps, Python's per-node overhead (computing ω) and the pruning step hide most of the saving. The paper's 0.01 s timer resolution adds ±16 pp of uncertainty to its own percentages.
- **Missing details.** See Q7. In addition:
  - the paper's text contradicts its tables in several places;
  - two Table 1 rows (60×60) appear swapped, and its pruning row is cumulative there (Parts 5.1 and 6.1);
  - the bidirectional A\* of ref. [30] is not described, so ours is a generic first-meeting version.
- **Our deviations.** D1–D12 apply identically to every planner, and none favours the proposed method.

**What our reproduction adds beyond the paper:**
1. **The adaptive search is not optimal.** The full estimate overestimates the true cost by up to 58.5% as printed (3.17× under V4). On random maps, the raw Adaptive A\* path is longer than optimal on 90 of 150 maps (Part 9).
2. **The length and smoothness gains come mostly from the pruning step.** A\* + pruning alone is usually as short or shorter (Part 9), and the paper's own Table 1 shows the same. The real benefit of the adaptive heuristic is **speed**.
3. **On published numbers alone**, Adaptive A\* improves on A\* more than Chen et al. and Zhang et al. do, so the paper's ranking holds.

## 12. Limitations and conclusion
**Limitations:**
- The maps are digitised from a 753×337 px screenshot. Maps 1 and 2 were not tuned. Map 3's block groups and two settings were calibrated (Part 10.2). Map 2 has not yet been cross-checked against the paper's high-resolution Fig. 7.
- Chen et al. and Zhang et al. were not re-implemented. Bidirectional A\* is a generic version, not ref. [30].
- Python timing is not directly comparable with MATLAB timing.
- Static grids only, as in the paper.

**Conclusion.** Re-implemented independently and run on the paper's own (digitised) maps, the baselines and each improvement match the published numbers closely. The full Adaptive A\*, implemented exactly as printed, is faster than A\*, expands fewer nodes, and gives shorter and smoother paths, as claimed. Its gains are smaller than published, however. The published node counts are consistent with the hybrid heuristic being used without the ½ of Eq. 7, which is a hypothesis we cannot prove. Our additional experiments show that the adaptive heuristic trades optimality for speed, and that redundant point deletion provides most of the path-quality improvement.